In [ ]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("Active Snowflake session acquired.")
except Exception as e:
    session = None
    print("Could not get active Snowflake session.")
    print(e)


In [ ]:
%%sql -r dataframe_25
select * from H_EDW.HS_REVELIO_EDW_V.V_POSTINGS_UNIFIED_INDIVIDUAL limit 100

In [ ]:
%%sql -r dataframe_1

USE DATABASE PC_FIVETRAN_DB;
USE SCHEMA DA_WORKSPACE;

SET SOURCE_TABLE = 'H_EDW.HS_REVELIO_EDW_V.V_POSTINGS_UNIFIED_INDIVIDUAL';

SET START_DATE = '2025-01-01';
SET END_DATE   = '2025-04-01';

SET GAP_DAYS = 10;

SET MERGE_EDGE_GAP_DAYS = 10;

SET MAX_MERGE_GROUP_SPAN_DAYS = 30;

SET SHORTER_JD_FULL_TOKEN_LIMIT = 3500;
SET STRAT_HEAD_CHAR_LIMIT = 1000;
SET STRAT_MID_SEGMENT_CHAR_LIMIT = 500;
SET STRAT_TAIL_CHAR_LIMIT = 1000;

-- Longer-side search uses the full cleaned JD.
-- No longer-JD cap is used in the decision path; only the shorter side is sampled when needed.

-- Stable bulk pair-scoring controls.
-- Bulk scoring uses token/word containment only.
SET MAX_ROWS_PER_PAIR_SCORING_CLUSTER = 150;
SET MAX_PAIRS_PER_PAIR_SCORING_CLUSTER = 5000;

-- Similarity thresholds.
-- Exact cleaned JD hash match -> MERGE_EXACT_JD.
-- Shorter-to-longer token containment >= 0.90 -> merge and use longer/best clean JD.
-- Shorter-to-longer token containment >= 0.75 and < 0.90 -> REVIEW_BUCKET.
-- Shorter-to-longer token containment < 0.75 -> KEEP_SEPARATE.
SET TOKEN_CONTAINMENT_MERGE = 0.90;
SET TOKEN_CONTAINMENT_REVIEW = 0.75;

-- Temporary helper tables.
SET BASE_TABLE                  = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_BASE_TC_STRAT';
SET DAY_TABLE                   = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_DAYS_TC_STRAT';
SET ORDERED_DAY_TABLE           = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ORDERED_DAYS_TC_STRAT';
SET DAY_CLUSTER_TABLE           = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_DAY_CLUSTERS_TC_STRAT';
SET ROW_CLUSTER_TABLE           = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ROWS_CLUSTERED_TC_STRAT';
SET CANDIDATE_METRICS_TABLE     = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_CANDIDATE_METRICS_TC_STRAT';
SET ORDERED_ROW_TABLE           = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ORDERED_ROWS_TC_STRAT';
SET PAIR_SCORE_TABLE            = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_PAIR_SCORES_TC_STRAT';
SET SKIPPED_PAIR_SCORING_CLUSTER_TABLE = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_SKIPPED_PAIR_SCORING_TC_STRAT';
SET MERGE_GROUP_TABLE           = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_GROUP_REP_ASSIGNMENT_TC_STRAT';
SET MERGE_GROUP_METRICS_TABLE   = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_GROUP_REP_METRICS_TC_STRAT';
SET SINGLETON_CONTEXT_TABLE     = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_SINGLETON_CONTEXT_TC_STRAT';

-- Persistent final output.
SET CANONICAL_TABLE             = 'PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_CANONICAL_OUTPUT_UPRCID_TITLETR_TC_GROUP_REP_10D';

-- Temporary review/lineage helpers for inspection during the session.
SET LINEAGE_TABLE               = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_LINEAGE_TC_GROUP_REP_STRAT_10D';
SET REVIEW_TABLE                = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_REVIEW_TC_GROUP_REP_STRAT_10D';


-- Persistent QA / handoff tables.
SET ROW_COUNT_QA_TABLE          = 'PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ROW_COUNT_QA_TC_GROUP_REP_10D';
SET BUCKET_SPOTCHECK_TABLE      = 'PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_BUCKET_SPOTCHECK_TC_GROUP_REP_10D';


-- Optional post-canonical translation enrichment.
-- Default is FALSE to avoid accidental Snowflake Cortex credit usage.
-- Set ENABLE_CANONICAL_TRANSLATION = TRUE after reviewing the candidate count cell.
SET ENABLE_CANONICAL_TRANSLATION = FALSE;
SET TRANSLATION_MIN_CLEANED_JD_LENGTH = 200;
SET TRANSLATION_MIN_NON_ASCII_COUNT = 20;
SET TRANSLATION_MIN_NON_ASCII_PCT = 5;
SET TRANSLATION_TEXT_CHAR_LIMIT = 16000;
SET CANONICAL_TRANSLATION_TABLE = 'PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_CANONICAL_TRANSLATION_UPRCID_TITLETR_10D';

SELECT 'Parameters set successfully - UPRCID + translated-title candidate-key version' AS STATUS;


In [ ]:
%%sql -r dataframe_2
-- SQL Cell 2: Create TEMP normalized base table.
-- Matching key = ULTIMATE_PARENT_RCID + COALESCE(TITLE_TRANSLATED, TITLE_RAW).
-- We keep p.* so all original source columns are available in final canonical output.
-- This version adds two token-scoring fields used by the decision logic:
--   1. SHORTER_SAMPLE_TEXT: full cleaned JD if <= 3,500 chars, otherwise stratified 3,500 chars.
--   2. LONGER_SEARCH_TEXT: full cleaned JD used as the longer-side containment search text.

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($BASE_TABLE) AS
WITH cleaned_base AS (
    SELECT
        p.*,

        TO_VARCHAR(p.JOB_ID) AS JOB_ID_STR,
        p.POST_DATE::DATE AS POST_DAY,
        p.REMOVE_DATE::DATE AS REMOVE_DAY,

        IFF(p.POST_DATE::DATE >= TO_DATE($START_DATE) AND p.POST_DATE::DATE < TO_DATE($END_DATE), TRUE, FALSE) AS IN_SCOPE_FLAG,

        -- Business-selected duplicate candidate key: ULTIMATE_PARENT_RCID + COALESCE(TITLE_TRANSLATED, TITLE_RAW).
        MD5(
            COALESCE(TRIM(TO_VARCHAR(p.ULTIMATE_PARENT_RCID)), '<<NULL_ULTIMATE_PARENT_RCID>>') || '|' ||
            COALESCE(
                NULLIF(TRIM(TO_VARCHAR(p.TITLE_TRANSLATED)), ''),
                NULLIF(TRIM(TO_VARCHAR(p.TITLE_RAW)), ''),
                '<<NULL_TITLE>>'
            )
        ) AS RAW_MATCH_KEY,

        COALESCE(TRIM(TO_VARCHAR(p.ULTIMATE_PARENT_RCID)), '<<NULL_ULTIMATE_PARENT_RCID>>') || ' | ' ||
        COALESCE(
            NULLIF(TRIM(TO_VARCHAR(p.TITLE_TRANSLATED)), ''),
            NULLIF(TRIM(TO_VARCHAR(p.TITLE_RAW)), ''),
            '<<NULL_TITLE>>'
        ) AS RAW_MATCH_KEY_TEXT,

        IFF(COALESCE(p.SOURCE_LINKEDIN, FALSE), 1, 0)
      + IFF(COALESCE(p.SOURCE_INDEED, FALSE), 1, 0)
      + IFF(COALESCE(p.SOURCE_COMPANY_SITES, FALSE), 1, 0)
      + IFF(COALESCE(p.SOURCE_STAFFINGFIRMS, FALSE), 1, 0)
      + IFF(COALESCE(p.SOURCE_51JOB, FALSE), 1, 0)
      + IFF(COALESCE(p.SOURCE_LIEPIN, FALSE), 1, 0)
      + IFF(COALESCE(p.SOURCE_ZHAOPIN, FALSE), 1, 0)
      + IFF(COALESCE(p.SOURCE_REGIONAL_AGGREGATORS, FALSE), 1, 0)
      + IFF(COALESCE(p.SOURCE_OTHER_AGGREGATORS, FALSE), 1, 0) AS SOURCE_FLAG_COUNT,

        COALESCE(
            NULLIF(
                RTRIM(
                    IFF(COALESCE(p.SOURCE_LINKEDIN, FALSE), 'LinkedIn|', '') ||
                    IFF(COALESCE(p.SOURCE_INDEED, FALSE), 'Indeed|', '') ||
                    IFF(COALESCE(p.SOURCE_COMPANY_SITES, FALSE), 'CompanySite|', '') ||
                    IFF(COALESCE(p.SOURCE_STAFFINGFIRMS, FALSE), 'StaffingFirms|', '') ||
                    IFF(COALESCE(p.SOURCE_51JOB, FALSE), '51Job|', '') ||
                    IFF(COALESCE(p.SOURCE_LIEPIN, FALSE), 'Liepin|', '') ||
                    IFF(COALESCE(p.SOURCE_ZHAOPIN, FALSE), 'Zhaopin|', '') ||
                    IFF(COALESCE(p.SOURCE_REGIONAL_AGGREGATORS, FALSE), 'RegionalAgg|', '') ||
                    IFF(COALESCE(p.SOURCE_OTHER_AGGREGATORS, FALSE), 'OtherAgg|', ''),
                    '|'
                ),
                ''
            ),
            'Unknown'
        ) AS SOURCE_SIGNATURE,

        p.DESCRIPTION AS DESCRIPTION_ORIGINAL,
        IFF(REGEXP_LIKE(COALESCE(p.DESCRIPTION, ''), '<[^>]+>'), TRUE, FALSE) AS DESCRIPTION_HAS_HTML_TAG,

        -- Clean JD by removing HTML tags and normalizing whitespace.
        LOWER(
            TRIM(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        COALESCE(p.DESCRIPTION, ''),
                        '<[^>]+>',
                        ' '
                    ),
                    '[[:space:]]+',
                    ' '
                )
            )
        ) AS CLEAN_DESCRIPTION

    FROM IDENTIFIER($SOURCE_TABLE) p
    WHERE p.POST_DATE >= TO_DATE($START_DATE)
      AND p.POST_DATE <  TO_DATE($END_DATE)
      AND p.POST_DATE IS NOT NULL
      AND (
            (p.TITLE_TRANSLATED IS NOT NULL AND TRIM(p.TITLE_TRANSLATED) <> '')
         OR (p.TITLE_RAW IS NOT NULL AND TRIM(p.TITLE_RAW) <> '')
      )
      AND p.COMPANY IS NOT NULL
      AND TRIM(p.COMPANY) <> ''
),

base_features AS (
    SELECT
        *,
        MD5(CLEAN_DESCRIPTION) AS CLEAN_DESCRIPTION_HASH,
        LENGTH(CLEAN_DESCRIPTION) AS CLEAN_DESCRIPTION_LENGTH,

        -- Shorter-side sample:
        -- If cleaned JD <= 3,500 chars, use the full text.
        -- If cleaned JD > 3,500 chars, sample across the document:
        -- first 1,000 + 500 around 33% + 500 around 50% + 500 around 67% + last 1,000.
        CASE
            WHEN LENGTH(CLEAN_DESCRIPTION) <= $SHORTER_JD_FULL_TOKEN_LIMIT THEN CLEAN_DESCRIPTION
            ELSE
                LEFT(CLEAN_DESCRIPTION, $STRAT_HEAD_CHAR_LIMIT)
                || ' ' ||
                SUBSTR(
                    CLEAN_DESCRIPTION,
                    GREATEST(1, FLOOR(LENGTH(CLEAN_DESCRIPTION) * 0.333333) - FLOOR($STRAT_MID_SEGMENT_CHAR_LIMIT / 2)),
                    $STRAT_MID_SEGMENT_CHAR_LIMIT
                )
                || ' ' ||
                SUBSTR(
                    CLEAN_DESCRIPTION,
                    GREATEST(1, FLOOR(LENGTH(CLEAN_DESCRIPTION) * 0.500000) - FLOOR($STRAT_MID_SEGMENT_CHAR_LIMIT / 2)),
                    $STRAT_MID_SEGMENT_CHAR_LIMIT
                )
                || ' ' ||
                SUBSTR(
                    CLEAN_DESCRIPTION,
                    GREATEST(1, FLOOR(LENGTH(CLEAN_DESCRIPTION) * 0.666667) - FLOOR($STRAT_MID_SEGMENT_CHAR_LIMIT / 2)),
                    $STRAT_MID_SEGMENT_CHAR_LIMIT
                )
                || ' ' ||
                RIGHT(CLEAN_DESCRIPTION, $STRAT_TAIL_CHAR_LIMIT)
        END AS SHORTER_SAMPLE_TEXT,

        -- Longer-side search text:
        -- Full cleaned JD is used so the shorter sample can match content anywhere in the longer description.
        CLEAN_DESCRIPTION AS LONGER_SEARCH_TEXT

    FROM cleaned_base
)

SELECT *
FROM base_features;

SELECT
    COUNT(*) AS BASE_ROWS,
    COUNT(DISTINCT JOB_ID) AS DISTINCT_JOB_IDS,
    COUNT(DISTINCT RAW_MATCH_KEY) AS DISTINCT_PARENT_TITLE_KEYS,
    COUNT_IF(CLEAN_DESCRIPTION IS NULL OR CLEAN_DESCRIPTION = '') AS BLANK_CLEAN_DESCRIPTION_ROWS,
    COUNT_IF(DESCRIPTION_HAS_HTML_TAG) AS ROWS_WITH_HTML_TAG,
    COUNT_IF(ULTIMATE_PARENT_RCID IS NULL) AS NULL_ULTIMATE_PARENT_RCID_ROWS,
    COUNT_IF(CLEAN_DESCRIPTION_LENGTH <= $SHORTER_JD_FULL_TOKEN_LIMIT) AS FULL_JD_TOKEN_ROWS,
    COUNT_IF(CLEAN_DESCRIPTION_LENGTH >  $SHORTER_JD_FULL_TOKEN_LIMIT) AS STRATIFIED_TOKEN_ROWS,
    ROUND(AVG(CLEAN_DESCRIPTION_LENGTH), 2) AS AVG_CLEAN_DESCRIPTION_LENGTH
FROM IDENTIFIER($BASE_TABLE);


In [ ]:
%%sql -r dataframe_3
-- SQL Cell 3: Create distinct posting-day table and order days within ULTIMATE_PARENT_RCID + COALESCE(TITLE_TRANSLATED, TITLE_RAW).

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($DAY_TABLE) AS
SELECT
    RAW_MATCH_KEY,
    POST_DAY,
    COUNT(*) AS RAW_ROWS_ON_DAY,
    COUNT(DISTINCT JOB_ID_STR) AS DISTINCT_JOB_IDS_ON_DAY
FROM IDENTIFIER($BASE_TABLE)
WHERE CLEAN_DESCRIPTION IS NOT NULL
  AND CLEAN_DESCRIPTION <> ''
GROUP BY RAW_MATCH_KEY, POST_DAY;

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($ORDERED_DAY_TABLE) AS
SELECT
    RAW_MATCH_KEY,
    POST_DAY,
    RAW_ROWS_ON_DAY,
    DISTINCT_JOB_IDS_ON_DAY,
    ROW_NUMBER() OVER (PARTITION BY RAW_MATCH_KEY ORDER BY POST_DAY) AS RN
FROM IDENTIFIER($DAY_TABLE);

SELECT
    COUNT(*) AS DAY_ROWS,
    COUNT(DISTINCT RAW_MATCH_KEY) AS DISTINCT_MATCH_KEYS
FROM IDENTIFIER($ORDERED_DAY_TABLE);


In [ ]:
%%sql -r dataframe_4
-- SQL Cell 4: Create rolling candidate clusters by ULTIMATE_PARENT_RCID + COALESCE(TITLE_TRANSLATED, TITLE_RAW).
-- A new candidate context starts only when the gap from the prior posting day is > 10 days.
-- This is not the final merge grouping.

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($DAY_CLUSTER_TABLE) AS
WITH RECURSIVE clustered_days (
    RAW_MATCH_KEY,
    POST_DAY,
    RN,
    CLUSTER_SEQUENCE,
    CLUSTER_START_DAY,
    PREV_POST_DAY
) AS (
    SELECT
        RAW_MATCH_KEY,
        POST_DAY,
        RN,
        1 AS CLUSTER_SEQUENCE,
        POST_DAY AS CLUSTER_START_DAY,
        POST_DAY AS PREV_POST_DAY
    FROM IDENTIFIER($ORDERED_DAY_TABLE)
    WHERE RN = 1

    UNION ALL

    SELECT
        d.RAW_MATCH_KEY,
        d.POST_DAY,
        d.RN,
        CASE
            WHEN DATEDIFF('DAY', c.PREV_POST_DAY, d.POST_DAY) > $GAP_DAYS
            THEN c.CLUSTER_SEQUENCE + 1
            ELSE c.CLUSTER_SEQUENCE
        END AS CLUSTER_SEQUENCE,
        CASE
            WHEN DATEDIFF('DAY', c.PREV_POST_DAY, d.POST_DAY) > $GAP_DAYS
            THEN d.POST_DAY
            ELSE c.CLUSTER_START_DAY
        END AS CLUSTER_START_DAY,
        d.POST_DAY AS PREV_POST_DAY
    FROM clustered_days c
    JOIN IDENTIFIER($ORDERED_DAY_TABLE) d
      ON c.RAW_MATCH_KEY = d.RAW_MATCH_KEY
     AND d.RN = c.RN + 1
)
SELECT
    RAW_MATCH_KEY,
    POST_DAY,
    RN,
    CLUSTER_SEQUENCE,
    CLUSTER_START_DAY,
    MD5(RAW_MATCH_KEY || '|' || TO_VARCHAR(CLUSTER_SEQUENCE)) AS CLUSTER_ID
FROM clustered_days;

SELECT
    COUNT(*) AS CLUSTER_DAY_ROWS,
    COUNT(DISTINCT CLUSTER_ID) AS CANDIDATE_CLUSTERS
FROM IDENTIFIER($DAY_CLUSTER_TABLE);


In [ ]:
%%sql -r dataframe_5
-- SQL Cell 5: Assign raw rows to the rolling candidate context.

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($ROW_CLUSTER_TABLE) AS
SELECT
    b.*,
    d.CLUSTER_ID,
    d.CLUSTER_SEQUENCE,
    d.CLUSTER_START_DAY,
    ROW_NUMBER() OVER (
        PARTITION BY d.CLUSTER_ID
        ORDER BY b.POST_DAY, b.JOB_ID_STR
    ) AS ROW_IN_CLUSTER
FROM IDENTIFIER($BASE_TABLE) b
JOIN IDENTIFIER($DAY_CLUSTER_TABLE) d
  ON b.RAW_MATCH_KEY = d.RAW_MATCH_KEY
 AND b.POST_DAY = d.POST_DAY
WHERE b.CLEAN_DESCRIPTION IS NOT NULL
  AND b.CLEAN_DESCRIPTION <> '';

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($CANDIDATE_METRICS_TABLE) AS
SELECT
    CLUSTER_ID,
    ANY_VALUE(RAW_MATCH_KEY) AS RAW_MATCH_KEY,
    ANY_VALUE(RAW_MATCH_KEY_TEXT) AS RAW_MATCH_KEY_TEXT,
    MIN(POST_DAY) AS CANDIDATE_CLUSTER_START_DATE,
    MAX(POST_DAY) AS CANDIDATE_CLUSTER_END_DATE,
    DATEDIFF('DAY', MIN(POST_DAY), MAX(POST_DAY)) AS CANDIDATE_CLUSTER_SPAN_DAYS,
    COUNT(*) AS RAW_ROWS_IN_CANDIDATE_CLUSTER,
    COUNT(DISTINCT JOB_ID_STR) AS CANDIDATE_CLUSTER_JOB_IDS,
    COUNT(DISTINCT COUNTRY) AS CANDIDATE_DISTINCT_COUNTRIES,
    COUNT(DISTINCT STATE) AS CANDIDATE_DISTINCT_STATES,
    LISTAGG(DISTINCT TO_VARCHAR(JOB_ID), ', ') WITHIN GROUP (ORDER BY TO_VARCHAR(JOB_ID)) AS CANDIDATE_RELEVANT_JOB_IDS,
    LISTAGG(DISTINCT NULLIF(TRIM(COUNTRY), ''), ', ') WITHIN GROUP (ORDER BY NULLIF(TRIM(COUNTRY), '')) AS CANDIDATE_ALL_COUNTRY_POSTED,
    LISTAGG(DISTINCT NULLIF(TRIM(STATE), ''), ', ') WITHIN GROUP (ORDER BY NULLIF(TRIM(STATE), '')) AS CANDIDATE_ALL_STATE_POSTED
FROM IDENTIFIER($ROW_CLUSTER_TABLE)
GROUP BY CLUSTER_ID;

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($ORDERED_ROW_TABLE) AS
SELECT
    r.*,
    ROW_NUMBER() OVER (PARTITION BY CLUSTER_ID ORDER BY POST_DAY, JOB_ID_STR) AS RN_IN_CLUSTER
FROM IDENTIFIER($ROW_CLUSTER_TABLE) r;

SELECT
    COUNT(*) AS ROWS_CLUSTERED,
    COUNT(DISTINCT CLUSTER_ID) AS CANDIDATE_CLUSTERS,
    COUNT_IF(CANDIDATE_CLUSTER_JOB_IDS > 1) AS MULTI_JOB_CANDIDATE_CLUSTERS
FROM IDENTIFIER($CANDIDATE_METRICS_TABLE);


In [ ]:
%%sql -r dataframe_6
-- SQL Cell 6: Pair-level JD scores inside candidate contexts.
-- Simplified token-only version using asymmetric shorter-to-longer token containment.
--
-- Decision logic retained:
--   Exact cleaned JD hash match -> MERGE_EXACT_JD.
--   Shorter-to-longer TOKEN_CONTAINMENT >= 0.90 -> MERGE_TOKEN_CONTAINMENT_USE_BEST_JD.
--   Shorter-to-longer TOKEN_CONTAINMENT >= 0.75 and < 0.90 -> REVIEW_BUCKET context.
--   Shorter-to-longer TOKEN_CONTAINMENT < 0.75 -> KEEP_SEPARATE context.
--
-- Removed from this version:
--   Longer-JD text cap, length ratio decisioning, Jaccard output, Jaro-Winkler, EditDistance.
--   These do not drive the final decision and previously added complexity/confusion.

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($SKIPPED_PAIR_SCORING_CLUSTER_TABLE) AS
WITH cluster_sizes AS (
    SELECT
        CLUSTER_ID,
        ANY_VALUE(RAW_MATCH_KEY) AS RAW_MATCH_KEY,
        ANY_VALUE(RAW_MATCH_KEY_TEXT) AS RAW_MATCH_KEY_TEXT,
        COUNT(*) AS ROWS_IN_CLUSTER,
        COUNT(DISTINCT JOB_ID_STR) AS DISTINCT_JOB_IDS_IN_CLUSTER,
        MIN(POST_DAY) AS CLUSTER_MIN_POST_DAY,
        MAX(POST_DAY) AS CLUSTER_MAX_POST_DAY
    FROM IDENTIFIER($ORDERED_ROW_TABLE)
    GROUP BY CLUSTER_ID
),
cluster_pair_estimate AS (
    SELECT
        *,
        ROWS_IN_CLUSTER * (ROWS_IN_CLUSTER - 1) / 2 AS MAX_THEORETICAL_PAIRS
    FROM cluster_sizes
)
SELECT
    *,
    CASE
        WHEN ROWS_IN_CLUSTER > $MAX_ROWS_PER_PAIR_SCORING_CLUSTER
            THEN 'Skipped bulk pair diagnostics because the candidate title/company cluster is too large for stable X-Small execution.'
        WHEN MAX_THEORETICAL_PAIRS > $MAX_PAIRS_PER_PAIR_SCORING_CLUSTER
            THEN 'Skipped bulk pair diagnostics because the candidate pair count is too large for stable X-Small execution.'
        ELSE NULL
    END AS SKIP_REASON
FROM cluster_pair_estimate
WHERE DISTINCT_JOB_IDS_IN_CLUSTER > 1
  AND (
        ROWS_IN_CLUSTER > $MAX_ROWS_PER_PAIR_SCORING_CLUSTER
     OR MAX_THEORETICAL_PAIRS > $MAX_PAIRS_PER_PAIR_SCORING_CLUSTER
  );

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($PAIR_SCORE_TABLE) AS

WITH cluster_sizes AS (
    SELECT
        CLUSTER_ID,
        COUNT(*) AS ROWS_IN_CLUSTER,
        COUNT(DISTINCT JOB_ID_STR) AS DISTINCT_JOB_IDS_IN_CLUSTER
    FROM IDENTIFIER($ORDERED_ROW_TABLE)
    GROUP BY CLUSTER_ID
),

scoring_rows AS (
    SELECT
        r.*,
        cs.ROWS_IN_CLUSTER,
        cs.DISTINCT_JOB_IDS_IN_CLUSTER
    FROM IDENTIFIER($ORDERED_ROW_TABLE) r
    JOIN cluster_sizes cs
      ON r.CLUSTER_ID = cs.CLUSTER_ID
    LEFT JOIN IDENTIFIER($SKIPPED_PAIR_SCORING_CLUSTER_TABLE) sk
      ON r.CLUSTER_ID = sk.CLUSTER_ID
    WHERE cs.DISTINCT_JOB_IDS_IN_CLUSTER > 1
      AND sk.CLUSTER_ID IS NULL
),

row_features AS (
    SELECT
        *,
        ARRAY_DISTINCT(STRTOK_TO_ARRAY(COALESCE(SHORTER_SAMPLE_TEXT, ''), ' ')) AS SHORTER_SIDE_TOKENS,
        ARRAY_SIZE(ARRAY_DISTINCT(STRTOK_TO_ARRAY(COALESCE(SHORTER_SAMPLE_TEXT, ''), ' '))) AS SHORTER_SIDE_TOKEN_COUNT,
        ARRAY_DISTINCT(STRTOK_TO_ARRAY(COALESCE(LONGER_SEARCH_TEXT, ''), ' ')) AS LONGER_SIDE_TOKENS
    FROM scoring_rows
),

candidate_pairs_pre AS (
    SELECT
        a.CLUSTER_ID,
        a.RAW_MATCH_KEY,
        a.RAW_MATCH_KEY_TEXT,

        a.JOB_ID_STR AS JOB_ID_1,
        b.JOB_ID_STR AS JOB_ID_2,
        a.POST_DAY AS POST_DAY_1,
        b.POST_DAY AS POST_DAY_2,
        DATEDIFF('DAY', a.POST_DAY, b.POST_DAY) AS PAIR_DATE_DIFF_DAYS,

        a.SOURCE_SIGNATURE AS SOURCE_SIGNATURE_1,
        b.SOURCE_SIGNATURE AS SOURCE_SIGNATURE_2,

        a.CLEAN_DESCRIPTION_HASH AS CLEAN_DESCRIPTION_HASH_1,
        b.CLEAN_DESCRIPTION_HASH AS CLEAN_DESCRIPTION_HASH_2,
        a.CLEAN_DESCRIPTION_LENGTH AS CLEAN_DESCRIPTION_LENGTH_1,
        b.CLEAN_DESCRIPTION_LENGTH AS CLEAN_DESCRIPTION_LENGTH_2,

        CASE
            WHEN a.CLEAN_DESCRIPTION_LENGTH <= b.CLEAN_DESCRIPTION_LENGTH THEN a.JOB_ID_STR
            ELSE b.JOB_ID_STR
        END AS SHORTER_JOB_ID,

        CASE
            WHEN a.CLEAN_DESCRIPTION_LENGTH <= b.CLEAN_DESCRIPTION_LENGTH THEN b.JOB_ID_STR
            ELSE a.JOB_ID_STR
        END AS LONGER_JOB_ID,

        CASE
            WHEN a.CLEAN_DESCRIPTION_LENGTH <= b.CLEAN_DESCRIPTION_LENGTH THEN a.SHORTER_SIDE_TOKENS
            ELSE b.SHORTER_SIDE_TOKENS
        END AS SHORTER_TOKENS,

        CASE
            WHEN a.CLEAN_DESCRIPTION_LENGTH <= b.CLEAN_DESCRIPTION_LENGTH THEN a.SHORTER_SIDE_TOKEN_COUNT
            ELSE b.SHORTER_SIDE_TOKEN_COUNT
        END AS SHORTER_TOKEN_COUNT,

        CASE
            WHEN a.CLEAN_DESCRIPTION_LENGTH <= b.CLEAN_DESCRIPTION_LENGTH THEN b.LONGER_SIDE_TOKENS
            ELSE a.LONGER_SIDE_TOKENS
        END AS LONGER_TOKENS,

        a.ROWS_IN_CLUSTER,
        a.DISTINCT_JOB_IDS_IN_CLUSTER,
        COUNT(*) OVER (PARTITION BY a.CLUSTER_ID) AS CANDIDATE_PAIRS_IN_CLUSTER

    FROM row_features a
    JOIN row_features b
      ON a.CLUSTER_ID = b.CLUSTER_ID
     AND a.RN_IN_CLUSTER < b.RN_IN_CLUSTER
     -- Pair scores are computed within 30 days because a row can compare to the group representative within the max group span.
     AND DATEDIFF('DAY', a.POST_DAY, b.POST_DAY) <= $MAX_MERGE_GROUP_SPAN_DAYS
),

candidate_pairs AS (
    SELECT *
    FROM candidate_pairs_pre
    WHERE CANDIDATE_PAIRS_IN_CLUSTER <= $MAX_PAIRS_PER_PAIR_SCORING_CLUSTER
),

token_scored AS (
    SELECT
        *,
        ARRAY_INTERSECTION(SHORTER_TOKENS, LONGER_TOKENS) AS TOKEN_INTERSECTION
    FROM candidate_pairs
),

final_scored AS (
    SELECT
        CLUSTER_ID,
        RAW_MATCH_KEY,
        RAW_MATCH_KEY_TEXT,

        JOB_ID_1,
        JOB_ID_2,
        POST_DAY_1,
        POST_DAY_2,
        PAIR_DATE_DIFF_DAYS,

        SOURCE_SIGNATURE_1,
        SOURCE_SIGNATURE_2,

        CLEAN_DESCRIPTION_HASH_1,
        CLEAN_DESCRIPTION_HASH_2,
        CLEAN_DESCRIPTION_LENGTH_1,
        CLEAN_DESCRIPTION_LENGTH_2,

        SHORTER_JOB_ID,
        LONGER_JOB_ID,
        SHORTER_TOKEN_COUNT,
        ARRAY_SIZE(TOKEN_INTERSECTION) AS TOKEN_OVERLAP_COUNT,
        ARRAY_SIZE(TOKEN_INTERSECTION)::FLOAT / NULLIF(SHORTER_TOKEN_COUNT, 0) AS TOKEN_CONTAINMENT_SCORE,

        ROWS_IN_CLUSTER,
        DISTINCT_JOB_IDS_IN_CLUSTER,
        CANDIDATE_PAIRS_IN_CLUSTER

    FROM token_scored
)

SELECT
    *,

    CASE
        WHEN CLEAN_DESCRIPTION_HASH_1 = CLEAN_DESCRIPTION_HASH_2
            THEN 'MERGE_EXACT_JD'
        WHEN TOKEN_CONTAINMENT_SCORE >= $TOKEN_CONTAINMENT_MERGE
            THEN 'MERGE_TOKEN_CONTAINMENT_USE_BEST_JD'
        ELSE NULL
    END AS PAIR_MERGE_ACTION,

    CASE
        WHEN CLEAN_DESCRIPTION_HASH_1 = CLEAN_DESCRIPTION_HASH_2
          OR TOKEN_CONTAINMENT_SCORE >= $TOKEN_CONTAINMENT_MERGE
        THEN TRUE
        ELSE FALSE
    END AS PAIR_JD_MERGE_ELIGIBLE,

    CASE
        WHEN TOKEN_CONTAINMENT_SCORE >= $TOKEN_CONTAINMENT_REVIEW
        THEN TRUE
        ELSE FALSE
    END AS PAIR_REVIEW_ELIGIBLE

FROM final_scored;

SELECT
    COUNT(*) AS PAIRS_SCORED,
    COUNT_IF(PAIR_JD_MERGE_ELIGIBLE) AS JD_MERGE_ELIGIBLE_PAIRS,
    COUNT_IF(PAIR_REVIEW_ELIGIBLE AND NOT PAIR_JD_MERGE_ELIGIBLE) AS REVIEW_ONLY_PAIRS,
    (SELECT COUNT(*) FROM IDENTIFIER($SKIPPED_PAIR_SCORING_CLUSTER_TABLE)) AS SKIPPED_LARGE_DIAGNOSTIC_CLUSTERS
FROM IDENTIFIER($PAIR_SCORE_TABLE);


In [ ]:
%%sql -r dataframe_7
/* 
SQL Cell 7A0: Hybrid assignment pre-step

Purpose:
1. Create assignment/checkpoint tables.z
2. Handle 1-row clusters directly in SQL.
3. Handle 2-row clusters directly in SQL.
4. Queue only 3+ row clusters for Python active-group processing.

Why:
This avoids sending all ~19M rows into Python. Python should only process clusters
where active group-representative logic is actually needed.
*/

SET ACTIVE_ASSIGNMENT_STAGE = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ACTIVE_GROUP_ASSIGNMENTS_STAGE';
SET ACTIVE_CLUSTER_QUEUE    = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ACTIVE_GROUP_CLUSTER_QUEUE';
SET ACTIVE_CLUSTER_DONE     = 'PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ACTIVE_GROUP_CLUSTER_DONE';

-- Reset assignment stage.
CREATE OR REPLACE TRANSIENT TABLE IDENTIFIER($ACTIVE_ASSIGNMENT_STAGE) (
    CLUSTER_ID STRING,
    RN_IN_CLUSTER NUMBER,
    MERGE_GROUP_SEQUENCE NUMBER,
    JOINED_TO_PREVIOUS_GROUP BOOLEAN,
    JOIN_ACTION STRING,
    JOIN_REASON STRING
);

-- Checkpoint table for Python 3+ cluster processing.
CREATE OR REPLACE TRANSIENT TABLE IDENTIFIER($ACTIVE_CLUSTER_DONE) (
    CLUSTER_ID STRING,
    COMPLETED_AT TIMESTAMP_NTZ
);

-- Cluster size table.
CREATE OR REPLACE TEMPORARY TABLE TMP_CLUSTER_SIZE AS
SELECT
    CLUSTER_ID,
    COUNT(*) AS ROWS_IN_CLUSTER,
    COUNT(DISTINCT JOB_ID_STR) AS DISTINCT_JOB_IDS_IN_CLUSTER,
    MIN(POST_DAY) AS CLUSTER_START_DAY,
    MAX(POST_DAY) AS CLUSTER_END_DAY
FROM IDENTIFIER($ORDERED_ROW_TABLE)
GROUP BY CLUSTER_ID;

-- 1) Single-row clusters: assign directly.
INSERT INTO IDENTIFIER($ACTIVE_ASSIGNMENT_STAGE)
SELECT
    r.CLUSTER_ID,
    r.RN_IN_CLUSTER,
    1 AS MERGE_GROUP_SEQUENCE,
    FALSE AS JOINED_TO_PREVIOUS_GROUP,
    'GROUP_ROOT' AS JOIN_ACTION,
    'Single-row candidate context; no duplicate grouping required.' AS JOIN_REASON
FROM IDENTIFIER($ORDERED_ROW_TABLE) r
JOIN TMP_CLUSTER_SIZE s
  ON r.CLUSTER_ID = s.CLUSTER_ID
WHERE s.ROWS_IN_CLUSTER = 1;

-- 2) Build two-row cluster scoring helper.
CREATE OR REPLACE TEMPORARY TABLE TMP_TWO_ROW_SCORED AS
WITH two_row_clusters AS (
    SELECT
        r.*
    FROM IDENTIFIER($ORDERED_ROW_TABLE) r
    JOIN TMP_CLUSTER_SIZE s
      ON r.CLUSTER_ID = s.CLUSTER_ID
    WHERE s.ROWS_IN_CLUSTER = 2
),

two_row_enriched AS (
    SELECT
        r.*,

        MAX(CASE WHEN RN_IN_CLUSTER = 1 THEN JOB_ID_STR END) OVER (
            PARTITION BY CLUSTER_ID
        ) AS JOB_ID_1,

        MAX(CASE WHEN RN_IN_CLUSTER = 2 THEN JOB_ID_STR END) OVER (
            PARTITION BY CLUSTER_ID
        ) AS JOB_ID_2,

        MAX(CASE WHEN RN_IN_CLUSTER = 1 THEN POST_DAY END) OVER (
            PARTITION BY CLUSTER_ID
        ) AS POST_DAY_1,

        MAX(CASE WHEN RN_IN_CLUSTER = 2 THEN POST_DAY END) OVER (
            PARTITION BY CLUSTER_ID
        ) AS POST_DAY_2

    FROM two_row_clusters r
)

SELECT
    r.*,
    ps.PAIR_JD_MERGE_ELIGIBLE,
    ps.PAIR_MERGE_ACTION
FROM two_row_enriched r
LEFT JOIN IDENTIFIER($PAIR_SCORE_TABLE) ps
  ON r.CLUSTER_ID = ps.CLUSTER_ID
 AND r.JOB_ID_1 = ps.JOB_ID_1
 AND r.JOB_ID_2 = ps.JOB_ID_2;

-- 3) Two-row clusters: assign directly using pair score.
INSERT INTO IDENTIFIER($ACTIVE_ASSIGNMENT_STAGE)
SELECT
    CLUSTER_ID,
    RN_IN_CLUSTER,

    CASE
        WHEN RN_IN_CLUSTER = 1 THEN 1

        WHEN COALESCE(PAIR_JD_MERGE_ELIGIBLE, FALSE) = TRUE
         AND DATEDIFF('DAY', POST_DAY_1, POST_DAY_2) <= $MERGE_EDGE_GAP_DAYS
         AND DATEDIFF('DAY', POST_DAY_1, POST_DAY_2) <= $MAX_MERGE_GROUP_SPAN_DAYS
            THEN 1

        ELSE 2
    END AS MERGE_GROUP_SEQUENCE,

    CASE
        WHEN RN_IN_CLUSTER = 2
         AND COALESCE(PAIR_JD_MERGE_ELIGIBLE, FALSE) = TRUE
         AND DATEDIFF('DAY', POST_DAY_1, POST_DAY_2) <= $MERGE_EDGE_GAP_DAYS
         AND DATEDIFF('DAY', POST_DAY_1, POST_DAY_2) <= $MAX_MERGE_GROUP_SPAN_DAYS
            THEN TRUE
        ELSE FALSE
    END AS JOINED_TO_PREVIOUS_GROUP,

    CASE
        WHEN RN_IN_CLUSTER = 1
            THEN 'GROUP_ROOT'

        WHEN COALESCE(PAIR_JD_MERGE_ELIGIBLE, FALSE) = TRUE
         AND DATEDIFF('DAY', POST_DAY_1, POST_DAY_2) <= $MERGE_EDGE_GAP_DAYS
         AND DATEDIFF('DAY', POST_DAY_1, POST_DAY_2) <= $MAX_MERGE_GROUP_SPAN_DAYS
            THEN PAIR_MERGE_ACTION

        ELSE 'GROUP_ROOT'
    END AS JOIN_ACTION,

    CASE
        WHEN RN_IN_CLUSTER = 1
            THEN 'First row in two-row candidate context starts the merge group.'

        WHEN COALESCE(PAIR_JD_MERGE_ELIGIBLE, FALSE) = TRUE
         AND DATEDIFF('DAY', POST_DAY_1, POST_DAY_2) <= $MERGE_EDGE_GAP_DAYS
         AND DATEDIFF('DAY', POST_DAY_1, POST_DAY_2) <= $MAX_MERGE_GROUP_SPAN_DAYS
            THEN 'Second row joined first row because pair is within date rules and JD/token containment is merge eligible.'

        ELSE 'Second row did not merge; started separate group because date or JD similarity rule failed.'
    END AS JOIN_REASON

FROM TMP_TWO_ROW_SCORED;

-- 4) Queue only 3+ row clusters for Python active-group logic.
CREATE OR REPLACE TRANSIENT TABLE IDENTIFIER($ACTIVE_CLUSTER_QUEUE) AS
SELECT
    CLUSTER_ID,
    ROW_NUMBER() OVER (ORDER BY CLUSTER_ID) AS CLUSTER_SEQ,
    ROWS_IN_CLUSTER,
    DISTINCT_JOB_IDS_IN_CLUSTER,
    CLUSTER_START_DAY,
    CLUSTER_END_DAY
FROM TMP_CLUSTER_SIZE
WHERE ROWS_IN_CLUSTER > 2;

-- 5) Validation summary.
SELECT
    'ASSIGNED_ROWS_FROM_SQL_1_AND_2_ROW_CLUSTERS' AS CHECKPOINT,
    COUNT(*) AS VALUE
FROM IDENTIFIER($ACTIVE_ASSIGNMENT_STAGE)

UNION ALL

SELECT
    'CLUSTERS_LEFT_FOR_PYTHON_3_PLUS' AS CHECKPOINT,
    COUNT(*) AS VALUE
FROM IDENTIFIER($ACTIVE_CLUSTER_QUEUE)

UNION ALL

SELECT
    'ROWS_LEFT_FOR_PYTHON_3_PLUS' AS CHECKPOINT,
    COALESCE(SUM(ROWS_IN_CLUSTER), 0) AS VALUE
FROM IDENTIFIER($ACTIVE_CLUSTER_QUEUE);

In [ ]:
# Python Cell 7A: Resume active group processing for 3+ row clusters
#
# IMPORTANT:
# - Do NOT rerun 7A0 before this.
# - This cell resumes from ACTIVE_CLUSTER_DONE.
# - Completed clusters are not rewritten.
# - The current unfinished batch/cluster is deleted from assignment stage before reprocessing.
#
# Fixes in this version:
# 1. Uses row-capped batch selection instead of fixed 10,000 clusters.
# 2. If the next cluster is very large, processes that single cluster alone.
# 3. For very large single clusters, fetches rows in RN chunks to avoid huge result-batch fetch failures.
# 4. Prints progress every 5,000 rows inside long batches.
# 5. Flushes assignments every 10,000 rows so partial work is visible.

import pandas as pd
from datetime import date, datetime

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    pass


# -----------------------------
# Snowflake variable helpers
# -----------------------------

def get_sf_var(name, default=None):
    try:
        return session.sql(f"SELECT ${name}").collect()[0][0]
    except Exception:
        return default


ORDERED_ROW_TABLE = get_sf_var(
    "ORDERED_ROW_TABLE",
    "PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ORDERED_ROWS_TC_STRAT"
)

ACTIVE_ASSIGNMENT_STAGE = get_sf_var(
    "ACTIVE_ASSIGNMENT_STAGE",
    "PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ACTIVE_GROUP_ASSIGNMENTS_STAGE"
)

ACTIVE_CLUSTER_QUEUE = get_sf_var(
    "ACTIVE_CLUSTER_QUEUE",
    "PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ACTIVE_GROUP_CLUSTER_QUEUE"
)

ACTIVE_CLUSTER_DONE = get_sf_var(
    "ACTIVE_CLUSTER_DONE",
    "PC_FIVETRAN_DB.DA_WORKSPACE.TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_ACTIVE_GROUP_CLUSTER_DONE"
)

MERGE_EDGE_GAP_DAYS = int(get_sf_var("MERGE_EDGE_GAP_DAYS", 10))
MAX_MERGE_GROUP_SPAN_DAYS = int(get_sf_var("MAX_MERGE_GROUP_SPAN_DAYS", 30))
TOKEN_CONTAINMENT_MERGE = float(get_sf_var("TOKEN_CONTAINMENT_MERGE", 0.90))

# Batch controls.
# These are execution controls only; they do not change merge rules.
BATCH_ROW_TARGET = 75000
BATCH_CLUSTER_LIMIT = 10000
LARGE_CLUSTER_SINGLE_BATCH_ROWS = 30000

# For a single very large cluster, fetch rows in smaller RN chunks.
LARGE_CLUSTER_ROW_FETCH_CHUNK = 5000

# Flush smaller batches so progress is saved more frequently.
ASSIGNMENT_WRITE_BATCH_SIZE = 10000

print("ORDERED_ROW_TABLE:", ORDERED_ROW_TABLE)
print("ACTIVE_ASSIGNMENT_STAGE:", ACTIVE_ASSIGNMENT_STAGE)
print("ACTIVE_CLUSTER_QUEUE:", ACTIVE_CLUSTER_QUEUE)
print("ACTIVE_CLUSTER_DONE:", ACTIVE_CLUSTER_DONE)
print("MERGE_EDGE_GAP_DAYS:", MERGE_EDGE_GAP_DAYS)
print("MAX_MERGE_GROUP_SPAN_DAYS:", MAX_MERGE_GROUP_SPAN_DAYS)
print("TOKEN_CONTAINMENT_MERGE:", TOKEN_CONTAINMENT_MERGE)
print("BATCH_ROW_TARGET:", BATCH_ROW_TARGET)
print("LARGE_CLUSTER_SINGLE_BATCH_ROWS:", LARGE_CLUSTER_SINGLE_BATCH_ROWS)
print("LARGE_CLUSTER_ROW_FETCH_CHUNK:", LARGE_CLUSTER_ROW_FETCH_CHUNK)


# -----------------------------
# Utility functions
# -----------------------------

def safe_bool(x):
    if x is None:
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return bool(x)
    return str(x).strip().upper() in ("TRUE", "1", "YES", "Y")


def safe_int(x, default=0):
    if x is None:
        return default
    try:
        if pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return int(x)
    except Exception:
        return default


def safe_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)


def sql_quote(value):
    return "'" + safe_str(value).replace("'", "''") + "'"


def to_py_date(x):
    if x is None:
        return None
    if isinstance(x, pd.Timestamp):
        return x.date()
    if isinstance(x, datetime):
        return x.date()
    if isinstance(x, date):
        return x
    return pd.to_datetime(x).date()


def days_between(d1, d2):
    d1 = to_py_date(d1)
    d2 = to_py_date(d2)
    if d1 is None or d2 is None:
        return 999999
    return (d2 - d1).days


def token_set(text):
    text = safe_str(text).lower()
    if not text:
        return set()
    return set(text.split())


def calc_similarity(row, group):
    """
    Compare current row to active group representative using shorter-to-longer token containment.

    Decision logic:
    - Exact cleaned JD hash match -> MERGE_EXACT_JD
    - Shorter-to-longer token containment >= 0.90 -> MERGE_TOKEN_CONTAINMENT_USE_BEST_JD

    Removed from decision path:
    - Length ratio
    - Jaccard
    - Jaro-Winkler
    - EditDistance
    """
    row_hash = safe_str(row.get("CLEAN_DESCRIPTION_HASH"))
    rep_hash = safe_str(group["rep_clean_description_hash"])

    if row_hash and rep_hash and row_hash == rep_hash:
        return {
            "eligible": True,
            "action": "MERGE_EXACT_JD",
            "containment": 1.0
        }

    row_len = safe_int(row.get("CLEAN_DESCRIPTION_LENGTH"))
    rep_len = safe_int(group["rep_clean_description_length"])

    row_short_tokens = token_set(row.get("SHORTER_SAMPLE_TEXT"))
    row_long_tokens = token_set(row.get("LONGER_SEARCH_TEXT"))

    rep_short_tokens = group["rep_shorter_tokens"]
    rep_long_tokens = group["rep_longer_tokens"]

    if row_len <= rep_len:
        shorter_tokens = row_short_tokens
        longer_tokens = rep_long_tokens
    else:
        shorter_tokens = rep_short_tokens
        longer_tokens = row_long_tokens

    short_count = len(shorter_tokens)

    if short_count == 0 or len(longer_tokens) == 0:
        containment = 0.0
    else:
        containment = len(shorter_tokens.intersection(longer_tokens)) / max(short_count, 1)

    if containment >= TOKEN_CONTAINMENT_MERGE:
        return {
            "eligible": True,
            "action": "MERGE_TOKEN_CONTAINMENT_USE_BEST_JD",
            "containment": containment
        }

    return {
        "eligible": False,
        "action": None,
        "containment": containment
    }


def is_better_rep(row, group):
    """
    Pick best JD representative for the merge group.

    Priority:
    1. Longer cleaned JD
    2. No HTML tags
    3. Company-site source
    4. Earlier post date
    5. Lower JOB_ID
    """
    row_len = safe_int(row.get("CLEAN_DESCRIPTION_LENGTH"))
    rep_len = safe_int(group["rep_clean_description_length"])

    if row_len != rep_len:
        return row_len > rep_len

    row_has_html = safe_bool(row.get("DESCRIPTION_HAS_HTML_TAG"))
    rep_has_html = safe_bool(group["rep_has_html"])

    if row_has_html != rep_has_html:
        return not row_has_html and rep_has_html

    row_company_site = safe_bool(row.get("SOURCE_COMPANY_SITES"))
    rep_company_site = safe_bool(group["rep_source_company_site"])

    if row_company_site != rep_company_site:
        return row_company_site and not rep_company_site

    row_date = to_py_date(row.get("POST_DAY"))
    rep_date = to_py_date(group["rep_post_day"])

    if row_date != rep_date:
        return row_date < rep_date

    return safe_str(row.get("JOB_ID_STR")) < safe_str(group["rep_job_id_str"])


def write_assignments(buffer):
    if not buffer:
        return

    pdf = pd.DataFrame(buffer)

    session.create_dataframe(pdf).write.mode("append").save_as_table(
        ACTIVE_ASSIGNMENT_STAGE
    )


def process_row(row, state, assignment_buffer, batch_num):
    cluster_id = safe_str(row["CLUSTER_ID"])

    if state["current_cluster_id"] != cluster_id:
        state["current_cluster_id"] = cluster_id
        state["active_groups"] = []
        state["next_group_sequence"] = 0

    row_date = to_py_date(row["POST_DAY"])
    row_hash = safe_str(row.get("CLEAN_DESCRIPTION_HASH"))
    row_len = safe_int(row.get("CLEAN_DESCRIPTION_LENGTH"))

    row_short_tokens = token_set(row.get("SHORTER_SAMPLE_TEXT"))
    row_long_tokens = token_set(row.get("LONGER_SEARCH_TEXT"))

    # Drop groups that are no longer active.
    state["active_groups"] = [
        g for g in state["active_groups"]
        if days_between(g["latest_day"], row_date) <= MERGE_EDGE_GAP_DAYS
    ]

    best_candidate = None

    for g in state["active_groups"]:
        latest_gap = days_between(g["latest_day"], row_date)
        span_gap = days_between(g["start_day"], row_date)

        if latest_gap > MERGE_EDGE_GAP_DAYS:
            continue
        if span_gap > MAX_MERGE_GROUP_SPAN_DAYS:
            continue

        sim = calc_similarity(row, g)

        if not sim["eligible"]:
            continue

        action_priority = {
            "MERGE_EXACT_JD": 2,
            "MERGE_TOKEN_CONTAINMENT_USE_BEST_JD": 1
        }.get(sim["action"], 0)

        score_tuple = (
            action_priority,
            sim["containment"],
            -latest_gap,
            -span_gap,
            -g["sequence"]
        )

        if best_candidate is None or score_tuple > best_candidate["score_tuple"]:
            best_candidate = {
                "group": g,
                "similarity": sim,
                "score_tuple": score_tuple
            }

    if best_candidate is not None:
        g = best_candidate["group"]
        sim = best_candidate["similarity"]

        merge_group_sequence = g["sequence"]
        joined = True
        join_action = sim["action"]
        join_reason = (
            "Joined active merge group because row is within 10 days of group latest row, "
            "within 30 days of group start, and shorter-to-longer token containment matched "
            "the current group representative."
        )

        g["latest_day"] = row_date
        g["member_count"] += 1

        if is_better_rep(row, g):
            g["rep_job_id_str"] = safe_str(row.get("JOB_ID_STR"))
            g["rep_clean_description_hash"] = row_hash
            g["rep_clean_description_length"] = row_len
            g["rep_has_html"] = safe_bool(row.get("DESCRIPTION_HAS_HTML_TAG"))
            g["rep_source_company_site"] = safe_bool(row.get("SOURCE_COMPANY_SITES"))
            g["rep_post_day"] = row_date
            g["rep_shorter_tokens"] = row_short_tokens
            g["rep_longer_tokens"] = row_long_tokens

    else:
        state["next_group_sequence"] += 1
        merge_group_sequence = state["next_group_sequence"]
        joined = False
        join_action = "GROUP_ROOT"
        join_reason = (
            "Started new merge group because row did not match any active group representative "
            "within the 10-day rolling window and 30-day max group span."
        )

        state["active_groups"].append({
            "sequence": merge_group_sequence,
            "start_day": row_date,
            "latest_day": row_date,
            "rep_job_id_str": safe_str(row.get("JOB_ID_STR")),
            "rep_clean_description_hash": row_hash,
            "rep_clean_description_length": row_len,
            "rep_has_html": safe_bool(row.get("DESCRIPTION_HAS_HTML_TAG")),
            "rep_source_company_site": safe_bool(row.get("SOURCE_COMPANY_SITES")),
            "rep_post_day": row_date,
            "rep_shorter_tokens": row_short_tokens,
            "rep_longer_tokens": row_long_tokens,
            "member_count": 1
        })

    assignment_buffer.append({
        "CLUSTER_ID": cluster_id,
        "RN_IN_CLUSTER": safe_int(row.get("RN_IN_CLUSTER")),
        "MERGE_GROUP_SEQUENCE": merge_group_sequence,
        "JOINED_TO_PREVIOUS_GROUP": joined,
        "JOIN_ACTION": join_action,
        "JOIN_REASON": join_reason
    })


# -----------------------------
# Main resume loop
# -----------------------------

batch_num = 0

while True:
    batch_pdf = session.sql(f"""
    WITH todo AS (
        SELECT
            q.CLUSTER_ID,
            q.CLUSTER_SEQ,
            q.ROWS_IN_CLUSTER
        FROM {ACTIVE_CLUSTER_QUEUE} q
        LEFT JOIN {ACTIVE_CLUSTER_DONE} d
          ON q.CLUSTER_ID = d.CLUSTER_ID
        WHERE d.CLUSTER_ID IS NULL
        ORDER BY q.CLUSTER_SEQ
        LIMIT {BATCH_CLUSTER_LIMIT}
    ),

    first_large_cluster AS (
        -- If any large cluster appears in the next candidate set,
        -- process the earliest large cluster by itself.
        SELECT
            CLUSTER_ID,
            ROWS_IN_CLUSTER
        FROM todo
        WHERE ROWS_IN_CLUSTER >= {LARGE_CLUSTER_SINGLE_BATCH_ROWS}
        QUALIFY ROW_NUMBER() OVER (ORDER BY CLUSTER_SEQ) = 1
    ),

    ranked_normal AS (
        SELECT
            *,
            SUM(ROWS_IN_CLUSTER) OVER (
                ORDER BY CLUSTER_SEQ
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS CUM_ROWS
        FROM todo
        WHERE NOT EXISTS (
            SELECT 1 FROM first_large_cluster
        )
    ),

    row_capped_batch AS (
        SELECT
            CLUSTER_ID,
            ROWS_IN_CLUSTER
        FROM ranked_normal
        WHERE CUM_ROWS <= {BATCH_ROW_TARGET}
    ),

    fallback_first_cluster AS (
        -- Safety fallback if no row-capped cluster qualifies.
        SELECT
            CLUSTER_ID,
            ROWS_IN_CLUSTER
        FROM todo
        WHERE NOT EXISTS (
            SELECT 1 FROM first_large_cluster
        )
          AND NOT EXISTS (
            SELECT 1 FROM row_capped_batch
        )
        QUALIFY ROW_NUMBER() OVER (ORDER BY CLUSTER_SEQ) = 1
    ),

    selected AS (
        SELECT * FROM first_large_cluster
        UNION ALL
        SELECT * FROM row_capped_batch
        UNION ALL
        SELECT * FROM fallback_first_cluster
    )

    SELECT DISTINCT
        CLUSTER_ID,
        ROWS_IN_CLUSTER
    FROM selected
""").to_pandas()

    if batch_pdf.empty:
        print("No remaining 3+ row clusters to process.")
        break

    batch_num += 1
    clusters_in_batch = len(batch_pdf)
    rows_in_batch = int(batch_pdf["ROWS_IN_CLUSTER"].sum())
    max_cluster_rows = int(batch_pdf["ROWS_IN_CLUSTER"].max())

    print(
        f"Starting batch {batch_num}; "
        f"clusters: {clusters_in_batch:,}; "
        f"rows_in_batch: {rows_in_batch:,}; "
        f"max_cluster_rows: {max_cluster_rows:,}"
    )

    # Stage selected cluster IDs for the batch.
    session.create_dataframe(batch_pdf[["CLUSTER_ID"]]).write.mode("overwrite").save_as_table(
        "TMP_ACTIVE_GROUP_CLUSTER_BATCH",
        table_type="temporary"
    )

    # Clean any prior partial assignment rows for these uncompleted clusters.
    session.sql(f"""
        DELETE FROM {ACTIVE_ASSIGNMENT_STAGE}
        WHERE CLUSTER_ID IN (
            SELECT CLUSTER_ID FROM TMP_ACTIVE_GROUP_CLUSTER_BATCH
        )
    """).collect()

    assignment_buffer = []
    state = {
        "current_cluster_id": None,
        "active_groups": [],
        "next_group_sequence": 0
    }
    rows_processed_in_batch = 0

    # If the batch is exactly one large cluster, fetch it in RN chunks.
    if clusters_in_batch == 1 and max_cluster_rows >= LARGE_CLUSTER_SINGLE_BATCH_ROWS:
        cluster_id = batch_pdf["CLUSTER_ID"].iloc[0]
        total_rows = max_cluster_rows

        print(
            f"Batch {batch_num}: processing one large cluster "
            f"{cluster_id} with {total_rows:,} rows in RN chunks..."
        )

        rn_start = 1

        while rn_start <= total_rows:
            rn_end = rn_start + LARGE_CLUSTER_ROW_FETCH_CHUNK - 1

            query = f"""
                SELECT
                    CLUSTER_ID,
                    RN_IN_CLUSTER,
                    JOB_ID_STR,
                    POST_DAY,
                    CLEAN_DESCRIPTION_HASH,
                    CLEAN_DESCRIPTION_LENGTH,
                    DESCRIPTION_HAS_HTML_TAG,
                    SOURCE_COMPANY_SITES,
                    SHORTER_SAMPLE_TEXT,
                    LONGER_SEARCH_TEXT
                FROM {ORDERED_ROW_TABLE}
                WHERE CLUSTER_ID = {sql_quote(cluster_id)}
                  AND RN_IN_CLUSTER BETWEEN {rn_start} AND {rn_end}
                ORDER BY RN_IN_CLUSTER
            """

            chunk_pdf = session.sql(query).to_pandas()

            for row in chunk_pdf.to_dict("records"):
                process_row(row, state, assignment_buffer, batch_num)
                rows_processed_in_batch += 1

                if rows_processed_in_batch % 5000 == 0:
                    print(
                        f"Batch {batch_num}: processed "
                        f"{rows_processed_in_batch:,} rows in current large cluster..."
                    )

                if len(assignment_buffer) >= ASSIGNMENT_WRITE_BATCH_SIZE:
                    write_assignments(assignment_buffer)
                    assignment_buffer = []

            rn_start = rn_end + 1

    else:
        # Normal row-capped batch.
        query = f"""
            SELECT
                r.CLUSTER_ID,
                r.RN_IN_CLUSTER,
                r.JOB_ID_STR,
                r.POST_DAY,
                r.CLEAN_DESCRIPTION_HASH,
                r.CLEAN_DESCRIPTION_LENGTH,
                r.DESCRIPTION_HAS_HTML_TAG,
                r.SOURCE_COMPANY_SITES,
                r.SHORTER_SAMPLE_TEXT,
                r.LONGER_SEARCH_TEXT
            FROM {ORDERED_ROW_TABLE} r
            JOIN TMP_ACTIVE_GROUP_CLUSTER_BATCH b
              ON r.CLUSTER_ID = b.CLUSTER_ID
            ORDER BY
                r.CLUSTER_ID,
                r.RN_IN_CLUSTER
        """

        for rows_pdf in session.sql(query).to_pandas_batches():
            for row in rows_pdf.to_dict("records"):
                process_row(row, state, assignment_buffer, batch_num)
                rows_processed_in_batch += 1

                if rows_processed_in_batch % 5000 == 0:
                    print(
                        f"Batch {batch_num}: processed "
                        f"{rows_processed_in_batch:,} rows in current batch..."
                    )

                if len(assignment_buffer) >= ASSIGNMENT_WRITE_BATCH_SIZE:
                    write_assignments(assignment_buffer)
                    assignment_buffer = []

    # Final flush for the batch.
    write_assignments(assignment_buffer)

    # Mark selected clusters done after successful batch completion.
    session.sql(f"""
        INSERT INTO {ACTIVE_CLUSTER_DONE}
        SELECT
            CLUSTER_ID,
            CURRENT_TIMESTAMP()
        FROM TMP_ACTIVE_GROUP_CLUSTER_BATCH
    """).collect()

    print(
        f"Finished batch {batch_num}; "
        f"clusters completed: {clusters_in_batch:,}; "
        f"rows processed: {rows_processed_in_batch:,}"
    )

print("Python 7A complete.")

In [ ]:
%%sql -r dataframe_9
-- SQL Cell 7B: Build MERGE_GROUP_TABLE from staged assignments.
--
-- Prerequisites:
--   SQL Cell 7A0 has populated ACTIVE_ASSIGNMENT_STAGE for 1-row / 2-row clusters.
--   Python Cell 7A has appended assignments for 3+ row clusters.
--
-- Purpose:
--   Join the assignment stage back to ORDERED_ROW_TABLE so downstream cells
--   can use all original row fields plus merge group metadata.

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($MERGE_GROUP_TABLE) AS

WITH assignment_dedup AS (
    SELECT
        CLUSTER_ID,
        RN_IN_CLUSTER,
        MERGE_GROUP_SEQUENCE,
        JOINED_TO_PREVIOUS_GROUP,
        JOIN_ACTION,
        JOIN_REASON
    FROM IDENTIFIER($ACTIVE_ASSIGNMENT_STAGE)
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY CLUSTER_ID, RN_IN_CLUSTER
        ORDER BY MERGE_GROUP_SEQUENCE
    ) = 1
),

joined AS (
    SELECT
        r.*,

        a.MERGE_GROUP_SEQUENCE,
        MD5(a.CLUSTER_ID || '|' || TO_VARCHAR(a.MERGE_GROUP_SEQUENCE)) AS MERGE_GROUP_ID,

        a.JOINED_TO_PREVIOUS_GROUP,
        a.JOIN_ACTION,
        a.JOIN_REASON

    FROM IDENTIFIER($ORDERED_ROW_TABLE) r
    JOIN assignment_dedup a
      ON r.CLUSTER_ID = a.CLUSTER_ID
     AND r.RN_IN_CLUSTER = a.RN_IN_CLUSTER
),

group_enriched AS (
    SELECT
        *,

        MIN(POST_DAY) OVER (
            PARTITION BY MERGE_GROUP_ID
        ) AS GROUP_START_DAY,

        MAX(POST_DAY) OVER (
            PARTITION BY MERGE_GROUP_ID
        ) AS GROUP_LATEST_DAY,

        FIRST_VALUE(JOB_ID_STR) OVER (
            PARTITION BY MERGE_GROUP_ID
            ORDER BY
                CLEAN_DESCRIPTION_LENGTH DESC,
                DESCRIPTION_HAS_HTML_TAG ASC,
                COALESCE(SOURCE_COMPANY_SITES, FALSE) DESC,
                POST_DAY ASC,
                JOB_ID_STR ASC
        ) AS GROUP_REP_JOB_ID_STR,

        FIRST_VALUE(CLEAN_DESCRIPTION_HASH) OVER (
            PARTITION BY MERGE_GROUP_ID
            ORDER BY
                CLEAN_DESCRIPTION_LENGTH DESC,
                DESCRIPTION_HAS_HTML_TAG ASC,
                COALESCE(SOURCE_COMPANY_SITES, FALSE) DESC,
                POST_DAY ASC,
                JOB_ID_STR ASC
        ) AS GROUP_REP_CLEAN_DESCRIPTION_HASH,

        FIRST_VALUE(CLEAN_DESCRIPTION_LENGTH) OVER (
            PARTITION BY MERGE_GROUP_ID
            ORDER BY
                CLEAN_DESCRIPTION_LENGTH DESC,
                DESCRIPTION_HAS_HTML_TAG ASC,
                COALESCE(SOURCE_COMPANY_SITES, FALSE) DESC,
                POST_DAY ASC,
                JOB_ID_STR ASC
        ) AS GROUP_REP_CLEAN_DESCRIPTION_LENGTH,

        FIRST_VALUE(DESCRIPTION_HAS_HTML_TAG) OVER (
            PARTITION BY MERGE_GROUP_ID
            ORDER BY
                CLEAN_DESCRIPTION_LENGTH DESC,
                DESCRIPTION_HAS_HTML_TAG ASC,
                COALESCE(SOURCE_COMPANY_SITES, FALSE) DESC,
                POST_DAY ASC,
                JOB_ID_STR ASC
        ) AS GROUP_REP_HAS_HTML_TAG,

        ROW_NUMBER() OVER (
            PARTITION BY MERGE_GROUP_ID
            ORDER BY
                POST_DAY ASC,
                JOB_ID_STR ASC
        ) AS EARLIEST_ROW_RANK,

        ROW_NUMBER() OVER (
            PARTITION BY MERGE_GROUP_ID
            ORDER BY
                CLEAN_DESCRIPTION_LENGTH DESC,
                DESCRIPTION_HAS_HTML_TAG ASC,
                COALESCE(SOURCE_COMPANY_SITES, FALSE) DESC,
                POST_DAY ASC,
                JOB_ID_STR ASC
        ) AS BEST_DESCRIPTION_ROW_RANK

    FROM joined
)

SELECT *
FROM group_enriched;

In [ ]:
%%sql -r dataframe_10
-- SQL Cell 7C: strict row-count validation between ORDERED_ROW_TABLE, assignment stage, and MERGE_GROUP_TABLE.

CREATE OR REPLACE TEMPORARY TABLE TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_PREFINAL_ROW_QA AS
WITH ordered_rows AS (
    SELECT COUNT(*) AS ROW_COUNT
    FROM IDENTIFIER($ORDERED_ROW_TABLE)
),
assignment_dedup AS (
    SELECT COUNT(*) AS ROW_COUNT
    FROM (
        SELECT CLUSTER_ID, RN_IN_CLUSTER
        FROM IDENTIFIER($ACTIVE_ASSIGNMENT_STAGE)
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY CLUSTER_ID, RN_IN_CLUSTER
            ORDER BY MERGE_GROUP_SEQUENCE
        ) = 1
    )
),
assignment_duplicates AS (
    SELECT COUNT(*) AS DUPLICATE_KEYS
    FROM (
        SELECT CLUSTER_ID, RN_IN_CLUSTER, COUNT(*) AS CNT
        FROM IDENTIFIER($ACTIVE_ASSIGNMENT_STAGE)
        GROUP BY CLUSTER_ID, RN_IN_CLUSTER
        HAVING COUNT(*) > 1
    )
),
merge_group_rows AS (
    SELECT COUNT(*) AS ROW_COUNT
    FROM IDENTIFIER($MERGE_GROUP_TABLE)
)
SELECT
    'ORDERED_ROWS_vs_ASSIGNMENT_STAGE' AS CHECK_NAME,
    (SELECT ROW_COUNT FROM ordered_rows) AS EXPECTED_VALUE,
    (SELECT ROW_COUNT FROM assignment_dedup) AS ACTUAL_VALUE,
    IFF((SELECT ROW_COUNT FROM ordered_rows) = (SELECT ROW_COUNT FROM assignment_dedup), 'PASS', 'FAIL') AS CHECK_STATUS
UNION ALL
SELECT
    'ORDERED_ROWS_vs_MERGE_GROUP_TABLE' AS CHECK_NAME,
    (SELECT ROW_COUNT FROM ordered_rows) AS EXPECTED_VALUE,
    (SELECT ROW_COUNT FROM merge_group_rows) AS ACTUAL_VALUE,
    IFF((SELECT ROW_COUNT FROM ordered_rows) = (SELECT ROW_COUNT FROM merge_group_rows), 'PASS', 'FAIL') AS CHECK_STATUS
UNION ALL
SELECT
    'DUPLICATE_ASSIGNMENT_KEYS' AS CHECK_NAME,
    0 AS EXPECTED_VALUE,
    (SELECT DUPLICATE_KEYS FROM assignment_duplicates) AS ACTUAL_VALUE,
    IFF((SELECT DUPLICATE_KEYS FROM assignment_duplicates) = 0, 'PASS', 'FAIL') AS CHECK_STATUS;

SELECT *
FROM TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_PREFINAL_ROW_QA
ORDER BY CHECK_NAME;

SELECT 1 / IFF(COUNT_IF(CHECK_STATUS = 'FAIL') = 0, 1, 0) AS STRICT_PREFINAL_ROW_COUNT_CHECK
FROM TMP_COSMOS_DEDUP_Q1_2025_UPRCID_TITLETR_PREFINAL_ROW_QA;


In [ ]:
%%sql -r dataframe_11
-- SQL Cell 8: Create merge-group metrics.
-- Source flags are OR'd across the actual merge group only.
-- ALL_RELEVANT_JOB_IDS contains only rows represented by the output row.
-- Large clusters skipped from bulk pair diagnostics are flagged as REVIEW_BUCKET if they remain singleton outputs.

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($SINGLETON_CONTEXT_TABLE) AS
SELECT
    JOB_ID_STR,
    MAX(TOKEN_CONTAINMENT_SCORE) AS MAX_RELATED_TOKEN_CONTAINMENT_SCORE,
    COUNT(*) AS RELATED_PAIR_COUNT,
    COUNT_IF(PAIR_REVIEW_ELIGIBLE) AS RELATED_REVIEW_PAIR_COUNT,
    COUNT_IF(PAIR_JD_MERGE_ELIGIBLE) AS RELATED_MERGE_PAIR_COUNT
FROM (
    SELECT JOB_ID_1 AS JOB_ID_STR, TOKEN_CONTAINMENT_SCORE, PAIR_REVIEW_ELIGIBLE, PAIR_JD_MERGE_ELIGIBLE
    FROM IDENTIFIER($PAIR_SCORE_TABLE)
    UNION ALL
    SELECT JOB_ID_2 AS JOB_ID_STR, TOKEN_CONTAINMENT_SCORE, PAIR_REVIEW_ELIGIBLE, PAIR_JD_MERGE_ELIGIBLE
    FROM IDENTIFIER($PAIR_SCORE_TABLE)
)
GROUP BY JOB_ID_STR;

CREATE OR REPLACE TEMPORARY TABLE IDENTIFIER($MERGE_GROUP_METRICS_TABLE) AS
WITH group_base AS (
    SELECT
        g.MERGE_GROUP_ID,
        ANY_VALUE(g.CLUSTER_ID) AS CLUSTER_ID,
        ANY_VALUE(g.RAW_MATCH_KEY) AS RAW_MATCH_KEY,
        ANY_VALUE(g.RAW_MATCH_KEY_TEXT) AS RAW_MATCH_KEY_TEXT,

        MIN(g.POST_DAY) AS MERGE_GROUP_START_DATE,
        MAX(g.POST_DAY) AS MERGE_GROUP_END_DATE,
        DATEDIFF('DAY', MIN(g.POST_DAY), MAX(g.POST_DAY)) AS MERGE_GROUP_SPAN_DAYS,

        COUNT(*) AS SOURCE_RECORD_COUNT,
        COUNT(DISTINCT g.JOB_ID_STR) AS JOB_IDS_IN_CLUSTER,
        COUNT(DISTINCT g.SOURCE_SIGNATURE) AS SOURCE_SIGNATURES_IN_CLUSTER,
        COUNT(DISTINCT g.COUNTRY) AS DISTINCT_COUNTRIES_IN_MERGE_GROUP,
        COUNT(DISTINCT g.STATE) AS DISTINCT_STATES_IN_MERGE_GROUP,

        LISTAGG(DISTINCT g.SOURCE_SIGNATURE, ', ') WITHIN GROUP (ORDER BY g.SOURCE_SIGNATURE) AS SOURCE_SIGNATURE_LIST,
        LISTAGG(DISTINCT TO_VARCHAR(g.JOB_ID), ', ') WITHIN GROUP (ORDER BY TO_VARCHAR(g.JOB_ID)) AS ALL_RELEVANT_JOB_IDS,
        LISTAGG(DISTINCT NULLIF(TRIM(g.COUNTRY), ''), ', ') WITHIN GROUP (ORDER BY NULLIF(TRIM(g.COUNTRY), '')) AS ALL_COUNTRY_POSTED,
        LISTAGG(DISTINCT NULLIF(TRIM(g.STATE), ''), ', ') WITHIN GROUP (ORDER BY NULLIF(TRIM(g.STATE), '')) AS ALL_STATE_POSTED,

        MAX(IFF(COALESCE(g.SOURCE_LINKEDIN, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_LINKEDIN,
        MAX(IFF(COALESCE(g.SOURCE_INDEED, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_INDEED,
        MAX(IFF(COALESCE(g.SOURCE_COMPANY_SITES, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_COMPANY_SITES,
        MAX(IFF(COALESCE(g.SOURCE_STAFFINGFIRMS, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_STAFFINGFIRMS,
        MAX(IFF(COALESCE(g.SOURCE_51JOB, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_51JOB,
        MAX(IFF(COALESCE(g.SOURCE_LIEPIN, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_LIEPIN,
        MAX(IFF(COALESCE(g.SOURCE_ZHAOPIN, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_ZHAOPIN,
        MAX(IFF(COALESCE(g.SOURCE_REGIONAL_AGGREGATORS, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_REGIONAL_AGGREGATORS,
        MAX(IFF(COALESCE(g.SOURCE_OTHER_AGGREGATORS, FALSE), 1, 0)) = 1 AS MERGED_SOURCE_OTHER_AGGREGATORS,

        COUNT_IF(g.JOIN_ACTION = 'MERGE_EXACT_JD') AS EXACT_JOIN_COUNT,
        COUNT_IF(g.JOIN_ACTION = 'MERGE_TOKEN_CONTAINMENT_USE_BEST_JD') AS TOKEN_CONTAINMENT_JOIN_COUNT,
        LISTAGG(DISTINCT g.JOIN_ACTION, ', ') AS JOIN_ACTIONS_IN_GROUP

    FROM IDENTIFIER($MERGE_GROUP_TABLE) g
    GROUP BY g.MERGE_GROUP_ID
),

earliest_rows AS (
    SELECT
        MERGE_GROUP_ID,
        JOB_ID_STR AS EARLIEST_JOB_ID_STR
    FROM IDENTIFIER($MERGE_GROUP_TABLE)
    WHERE EARLIEST_ROW_RANK = 1
),

best_desc_rows AS (
    SELECT
        MERGE_GROUP_ID,
        JOB_ID_STR AS BEST_DESCRIPTION_SOURCE_JOB_ID,
        CLEAN_DESCRIPTION AS BEST_AVAILABLE_CLEANED_JD,
        DESCRIPTION_ORIGINAL AS BEST_AVAILABLE_DESCRIPTION,
        CLEAN_DESCRIPTION_HASH AS BEST_AVAILABLE_CLEAN_DESCRIPTION_HASH,
        CLEAN_DESCRIPTION_LENGTH AS BEST_AVAILABLE_CLEAN_DESCRIPTION_LENGTH
    FROM IDENTIFIER($MERGE_GROUP_TABLE)
    WHERE BEST_DESCRIPTION_ROW_RANK = 1
),

skipped_diag AS (
    SELECT
        CLUSTER_ID,
        TRUE AS PAIR_DIAGNOSTIC_SKIPPED_FLAG,
        SKIP_REASON AS PAIR_DIAGNOSTIC_SKIP_REASON,
        ROWS_IN_CLUSTER AS PAIR_DIAGNOSTIC_SKIPPED_CLUSTER_ROWS,
        MAX_THEORETICAL_PAIRS AS PAIR_DIAGNOSTIC_SKIPPED_THEORETICAL_PAIRS
    FROM IDENTIFIER($SKIPPED_PAIR_SCORING_CLUSTER_TABLE)
)

SELECT
    gb.*,
    er.EARLIEST_JOB_ID_STR,
    bd.BEST_DESCRIPTION_SOURCE_JOB_ID,
    bd.BEST_AVAILABLE_CLEANED_JD,
    bd.BEST_AVAILABLE_DESCRIPTION,
    bd.BEST_AVAILABLE_CLEAN_DESCRIPTION_HASH,
    bd.BEST_AVAILABLE_CLEAN_DESCRIPTION_LENGTH,

    cm.CANDIDATE_CLUSTER_JOB_IDS,
    cm.CANDIDATE_RELEVANT_JOB_IDS,
    cm.CANDIDATE_ALL_COUNTRY_POSTED,
    cm.CANDIDATE_ALL_STATE_POSTED,

    sc.MAX_RELATED_TOKEN_CONTAINMENT_SCORE,
    sc.RELATED_PAIR_COUNT,
    sc.RELATED_REVIEW_PAIR_COUNT,
    sc.RELATED_MERGE_PAIR_COUNT,

    COALESCE(sd.PAIR_DIAGNOSTIC_SKIPPED_FLAG, FALSE) AS PAIR_DIAGNOSTIC_SKIPPED_FLAG,
    sd.PAIR_DIAGNOSTIC_SKIP_REASON,
    sd.PAIR_DIAGNOSTIC_SKIPPED_CLUSTER_ROWS,
    sd.PAIR_DIAGNOSTIC_SKIPPED_THEORETICAL_PAIRS,

    CASE
        WHEN gb.SOURCE_RECORD_COUNT > 1 AND gb.TOKEN_CONTAINMENT_JOIN_COUNT > 0
            THEN 'MERGE_TOKEN_CONTAINMENT_USE_BEST_JD'
        WHEN gb.SOURCE_RECORD_COUNT > 1 AND gb.EXACT_JOIN_COUNT > 0
            THEN 'MERGE_EXACT_JD'
        WHEN gb.SOURCE_RECORD_COUNT = 1 AND COALESCE(cm.CANDIDATE_CLUSTER_JOB_IDS, 1) = 1
            THEN 'NOTHING'
        WHEN gb.SOURCE_RECORD_COUNT = 1 AND COALESCE(sd.PAIR_DIAGNOSTIC_SKIPPED_FLAG, FALSE) = TRUE
            THEN 'REVIEW_BUCKET'
        WHEN gb.SOURCE_RECORD_COUNT = 1
         AND COALESCE(sc.MAX_RELATED_TOKEN_CONTAINMENT_SCORE, 0) >= $TOKEN_CONTAINMENT_REVIEW
            THEN 'REVIEW_BUCKET'
        ELSE 'KEEP_SEPARATE_MATERIAL_JD_CHANGE'
    END AS CANONICALIZATION_ACTION,

    CASE
        WHEN gb.SOURCE_RECORD_COUNT > 1 AND gb.TOKEN_CONTAINMENT_JOIN_COUNT > 0
            THEN 'Merged using group-representative rule; shorter-to-longer token containment was >= 0.90, so use the best/longest/cleanest JD.'
        WHEN gb.SOURCE_RECORD_COUNT > 1 AND gb.EXACT_JOIN_COUNT > 0
            THEN 'Merged using group-representative rule; cleaned JD matched exactly.'
        WHEN gb.SOURCE_RECORD_COUNT = 1 AND COALESCE(cm.CANDIDATE_CLUSTER_JOB_IDS, 1) = 1
            THEN 'No related duplicate candidate in this title/company candidate context.'
        WHEN gb.SOURCE_RECORD_COUNT = 1 AND COALESCE(sd.PAIR_DIAGNOSTIC_SKIPPED_FLAG, FALSE) = TRUE
            THEN 'Bulk pair diagnostics were skipped for this large title/company candidate context; row is kept separate but flagged for review instead of defaulting to keep-separate.'
        WHEN gb.SOURCE_RECORD_COUNT = 1
         AND COALESCE(sc.MAX_RELATED_TOKEN_CONTAINMENT_SCORE, 0) >= $TOKEN_CONTAINMENT_REVIEW
            THEN 'Related postings exist but did not qualify for auto-merge; send to review.'
        ELSE 'Related postings exist but JD similarity was too low; keep separate.'
    END AS CANONICALIZATION_REASON

FROM group_base gb
JOIN earliest_rows er
  ON gb.MERGE_GROUP_ID = er.MERGE_GROUP_ID
JOIN best_desc_rows bd
  ON gb.MERGE_GROUP_ID = bd.MERGE_GROUP_ID
LEFT JOIN IDENTIFIER($CANDIDATE_METRICS_TABLE) cm
  ON gb.CLUSTER_ID = cm.CLUSTER_ID
LEFT JOIN IDENTIFIER($SINGLETON_CONTEXT_TABLE) sc
  ON er.EARLIEST_JOB_ID_STR = sc.JOB_ID_STR
LEFT JOIN skipped_diag sd
  ON gb.CLUSTER_ID = sd.CLUSTER_ID;

SELECT
    CANONICALIZATION_ACTION,
    COUNT(*) AS OUTPUT_GROUPS,
    SUM(SOURCE_RECORD_COUNT) AS RAW_ROWS_REPRESENTED
FROM IDENTIFIER($MERGE_GROUP_METRICS_TABLE)
GROUP BY CANONICALIZATION_ACTION
ORDER BY RAW_ROWS_REPRESENTED DESC;


In [ ]:
%%sql -r dataframe_12
-- SQL Cell 9: Create final persistent canonical output table.
-- All original columns come from the earliest posted row in the merge group.
-- Source flags are OR'd across the actual merge group.
-- CLEANED_JD is selected from the best/longest/cleanest JD in the group.

CREATE OR REPLACE TABLE IDENTIFIER($CANONICAL_TABLE) AS
SELECT
    e.* EXCLUDE (
        JOB_ID_STR,
        POST_DAY,
        REMOVE_DAY,
        IN_SCOPE_FLAG,
        RAW_MATCH_KEY,
        RAW_MATCH_KEY_TEXT,
        SOURCE_FLAG_COUNT,
        SOURCE_SIGNATURE,
        DESCRIPTION_ORIGINAL,
        DESCRIPTION_HAS_HTML_TAG,
        CLEAN_DESCRIPTION,
        CLEAN_DESCRIPTION_HASH,
        CLEAN_DESCRIPTION_LENGTH,
        SHORTER_SAMPLE_TEXT,
        LONGER_SEARCH_TEXT,
        CLUSTER_ID,
        CLUSTER_SEQUENCE,
        CLUSTER_START_DAY,
        ROW_IN_CLUSTER,
        RN_IN_CLUSTER,
        MERGE_GROUP_SEQUENCE,
        MERGE_GROUP_ID,
        GROUP_START_DAY,
        GROUP_LATEST_DAY,
        GROUP_REP_JOB_ID_STR,
        GROUP_REP_CLEAN_DESCRIPTION_HASH,
        GROUP_REP_CLEAN_DESCRIPTION_LENGTH,
        GROUP_REP_HAS_HTML_TAG,
        JOINED_TO_PREVIOUS_GROUP,
        JOIN_ACTION,
        JOIN_REASON,
        EARLIEST_ROW_RANK,
        BEST_DESCRIPTION_ROW_RANK,
        SOURCE_LINKEDIN,
        SOURCE_INDEED,
        SOURCE_COMPANY_SITES,
        SOURCE_STAFFINGFIRMS,
        SOURCE_51JOB,
        SOURCE_LIEPIN,
        SOURCE_ZHAOPIN,
        SOURCE_REGIONAL_AGGREGATORS,
        SOURCE_OTHER_AGGREGATORS
    ),

    -- Source flags merged only across actual output group.
    mg.MERGED_SOURCE_LINKEDIN AS SOURCE_LINKEDIN,
    mg.MERGED_SOURCE_INDEED AS SOURCE_INDEED,
    mg.MERGED_SOURCE_COMPANY_SITES AS SOURCE_COMPANY_SITES,
    mg.MERGED_SOURCE_STAFFINGFIRMS AS SOURCE_STAFFINGFIRMS,
    mg.MERGED_SOURCE_51JOB AS SOURCE_51JOB,
    mg.MERGED_SOURCE_LIEPIN AS SOURCE_LIEPIN,
    mg.MERGED_SOURCE_ZHAOPIN AS SOURCE_ZHAOPIN,
    mg.MERGED_SOURCE_REGIONAL_AGGREGATORS AS SOURCE_REGIONAL_AGGREGATORS,
    mg.MERGED_SOURCE_OTHER_AGGREGATORS AS SOURCE_OTHER_AGGREGATORS,

    e.JOB_ID AS CANONICAL_JOB_ID,
    e.POST_DAY AS CANONICAL_POST_DATE,
    mg.BEST_DESCRIPTION_SOURCE_JOB_ID AS DESCRIPTION_SOURCE_JOB_ID,
    mg.CLUSTER_ID,
    mg.MERGE_GROUP_ID,
    mg.RAW_MATCH_KEY_TEXT,

    mg.CANONICALIZATION_ACTION,
    mg.CANONICALIZATION_REASON,

    CASE
        WHEN mg.CANONICALIZATION_ACTION = 'NOTHING' THEN 'NO_DUPLICATE'
        WHEN mg.CANONICALIZATION_ACTION IN ('MERGE_EXACT_JD', 'MERGE_TOKEN_CONTAINMENT_USE_BEST_JD') THEN 'MERGE'
        WHEN mg.CANONICALIZATION_ACTION = 'REVIEW_BUCKET' THEN 'REVIEW_BUCKET'
        ELSE 'KEEP_SEPARATE'
    END AS DEDUPE_CRITERIA,

    IFF(mg.CANONICALIZATION_ACTION LIKE 'MERGE%', TRUE, FALSE) AS AUTO_MERGE_FLAG,

    mg.SOURCE_RECORD_COUNT,
    mg.JOB_IDS_IN_CLUSTER,
    mg.SOURCE_SIGNATURES_IN_CLUSTER,
    mg.MERGE_GROUP_START_DATE,
    mg.MERGE_GROUP_END_DATE,
    mg.MERGE_GROUP_SPAN_DAYS,
    mg.SOURCE_SIGNATURE_LIST,

    mg.ALL_RELEVANT_JOB_IDS,
    mg.CANDIDATE_RELEVANT_JOB_IDS,
    mg.CANDIDATE_CLUSTER_JOB_IDS,

    mg.ALL_COUNTRY_POSTED,
    mg.ALL_STATE_POSTED,
    IFF(mg.DISTINCT_COUNTRIES_IN_MERGE_GROUP > 1, TRUE, FALSE) AS IS_MULTIPLE_COUNTRY_POSTING,
    IFF(mg.DISTINCT_STATES_IN_MERGE_GROUP > 1, TRUE, FALSE) AS IS_MULTIPLE_STATE_POSTING,

    mg.CANDIDATE_ALL_COUNTRY_POSTED,
    mg.CANDIDATE_ALL_STATE_POSTED,

    mg.BEST_AVAILABLE_DESCRIPTION AS SELECTED_DESCRIPTION,
    mg.BEST_AVAILABLE_CLEANED_JD AS CLEANED_JD,
    mg.BEST_AVAILABLE_CLEAN_DESCRIPTION_HASH,
    mg.BEST_AVAILABLE_CLEAN_DESCRIPTION_LENGTH,

    mg.EXACT_JOIN_COUNT,
    mg.TOKEN_CONTAINMENT_JOIN_COUNT,
    mg.JOIN_ACTIONS_IN_GROUP,


    mg.MAX_RELATED_TOKEN_CONTAINMENT_SCORE,
    mg.RELATED_PAIR_COUNT,
    mg.RELATED_REVIEW_PAIR_COUNT,
    mg.RELATED_MERGE_PAIR_COUNT,

    mg.PAIR_DIAGNOSTIC_SKIPPED_FLAG,
    mg.PAIR_DIAGNOSTIC_SKIP_REASON,
    mg.PAIR_DIAGNOSTIC_SKIPPED_CLUSTER_ROWS,
    mg.PAIR_DIAGNOSTIC_SKIPPED_THEORETICAL_PAIRS

FROM IDENTIFIER($MERGE_GROUP_METRICS_TABLE) mg
JOIN IDENTIFIER($MERGE_GROUP_TABLE) e
  ON mg.MERGE_GROUP_ID = e.MERGE_GROUP_ID
 AND e.EARLIEST_ROW_RANK = 1;

SELECT
    CANONICALIZATION_ACTION,
    DEDUPE_CRITERIA,
    COUNT(*) AS FINAL_ROWS,
    SUM(SOURCE_RECORD_COUNT) AS SOURCE_RECORDS_REPRESENTED,
    SUM(IFF(AUTO_MERGE_FLAG, SOURCE_RECORD_COUNT - 1, 0)) AS RAW_ROWS_REDUCED_BY_MERGE
FROM IDENTIFIER($CANONICAL_TABLE)
GROUP BY CANONICALIZATION_ACTION, DEDUPE_CRITERIA
ORDER BY FINAL_ROWS DESC;


In [ ]:
%%sql -r dataframe_13
-- SQL Cell 9B: strict final-output row-count validation and skipped-diagnostic review validation.

CREATE OR REPLACE TRANSIENT TABLE IDENTIFIER($ROW_COUNT_QA_TABLE) AS
WITH ordered_rows AS (
    SELECT COUNT(*) AS ROW_COUNT
    FROM IDENTIFIER($ORDERED_ROW_TABLE)
),
merge_group_rows AS (
    SELECT
        COUNT(*) AS ROW_COUNT,
        COUNT(DISTINCT MERGE_GROUP_ID) AS MERGE_GROUP_COUNT
    FROM IDENTIFIER($MERGE_GROUP_TABLE)
),
canonical_rows AS (
    SELECT
        COUNT(*) AS ROW_COUNT,
        SUM(SOURCE_RECORD_COUNT) AS SOURCE_RECORDS_REPRESENTED,
        COUNT_IF(CANONICALIZATION_ACTION IS NULL) AS NULL_ACTION_ROWS,
        COUNT_IF(DEDUPE_CRITERIA IS NULL) AS NULL_CRITERIA_ROWS,
        COUNT_IF(CLEANED_JD IS NULL OR TRIM(CLEANED_JD) = '') AS BLANK_CLEANED_JD_ROWS
    FROM IDENTIFIER($CANONICAL_TABLE)
),
skipped_singletons_not_review AS (
    SELECT COUNT(*) AS BAD_ROWS
    FROM IDENTIFIER($CANONICAL_TABLE)
    WHERE COALESCE(PAIR_DIAGNOSTIC_SKIPPED_FLAG, FALSE) = TRUE
      AND SOURCE_RECORD_COUNT = 1
      AND CANONICALIZATION_ACTION <> 'REVIEW_BUCKET'
)
SELECT
    'MERGE_GROUP_ROWS_vs_CANONICAL_SOURCE_RECORDS' AS CHECK_NAME,
    (SELECT ROW_COUNT FROM merge_group_rows) AS EXPECTED_VALUE,
    (SELECT SOURCE_RECORDS_REPRESENTED FROM canonical_rows) AS ACTUAL_VALUE,
    IFF((SELECT ROW_COUNT FROM merge_group_rows) = (SELECT SOURCE_RECORDS_REPRESENTED FROM canonical_rows), 'PASS', 'FAIL') AS CHECK_STATUS
UNION ALL
SELECT
    'ORDERED_ROWS_vs_CANONICAL_SOURCE_RECORDS' AS CHECK_NAME,
    (SELECT ROW_COUNT FROM ordered_rows) AS EXPECTED_VALUE,
    (SELECT SOURCE_RECORDS_REPRESENTED FROM canonical_rows) AS ACTUAL_VALUE,
    IFF((SELECT ROW_COUNT FROM ordered_rows) = (SELECT SOURCE_RECORDS_REPRESENTED FROM canonical_rows), 'PASS', 'FAIL') AS CHECK_STATUS
UNION ALL
SELECT
    'CANONICAL_ROWS_vs_MERGE_GROUPS' AS CHECK_NAME,
    (SELECT MERGE_GROUP_COUNT FROM merge_group_rows) AS EXPECTED_VALUE,
    (SELECT ROW_COUNT FROM canonical_rows) AS ACTUAL_VALUE,
    IFF((SELECT MERGE_GROUP_COUNT FROM merge_group_rows) = (SELECT ROW_COUNT FROM canonical_rows), 'PASS', 'FAIL') AS CHECK_STATUS
UNION ALL
SELECT
    'NULL_CANONICALIZATION_ACTION_ROWS' AS CHECK_NAME,
    0 AS EXPECTED_VALUE,
    (SELECT NULL_ACTION_ROWS FROM canonical_rows) AS ACTUAL_VALUE,
    IFF((SELECT NULL_ACTION_ROWS FROM canonical_rows) = 0, 'PASS', 'FAIL') AS CHECK_STATUS
UNION ALL
SELECT
    'NULL_DEDUPE_CRITERIA_ROWS' AS CHECK_NAME,
    0 AS EXPECTED_VALUE,
    (SELECT NULL_CRITERIA_ROWS FROM canonical_rows) AS ACTUAL_VALUE,
    IFF((SELECT NULL_CRITERIA_ROWS FROM canonical_rows) = 0, 'PASS', 'FAIL') AS CHECK_STATUS
UNION ALL
SELECT
    'BLANK_CLEANED_JD_ROWS' AS CHECK_NAME,
    0 AS EXPECTED_VALUE,
    (SELECT BLANK_CLEANED_JD_ROWS FROM canonical_rows) AS ACTUAL_VALUE,
    IFF((SELECT BLANK_CLEANED_JD_ROWS FROM canonical_rows) = 0, 'PASS', 'WARN') AS CHECK_STATUS
UNION ALL
SELECT
    'PAIR_DIAGNOSTIC_SKIPPED_SINGLETONS_NOT_REVIEW' AS CHECK_NAME,
    0 AS EXPECTED_VALUE,
    (SELECT BAD_ROWS FROM skipped_singletons_not_review) AS ACTUAL_VALUE,
    IFF((SELECT BAD_ROWS FROM skipped_singletons_not_review) = 0, 'PASS', 'FAIL') AS CHECK_STATUS;

SELECT *
FROM IDENTIFIER($ROW_COUNT_QA_TABLE)
ORDER BY CHECK_NAME;

SELECT 1 / IFF(COUNT_IF(CHECK_STATUS = 'FAIL') = 0, 1, 0) AS STRICT_FINAL_ROW_COUNT_CHECK
FROM IDENTIFIER($ROW_COUNT_QA_TABLE);


In [ ]:
%%sql -r dataframe_14
-- SQL Cell 9C: Count final canonical rows that are likely non-English translation candidates.
-- This cell does not call Snowflake Cortex and does not consume AI credits.

WITH scored AS (
    SELECT
        CANONICAL_JOB_ID,
        TITLE_RAW,
        TITLE_TRANSLATED,
        COMPANY,
        COUNTRY,
        CANONICALIZATION_ACTION,
        DEDUPE_CRITERIA,
        LENGTH(CLEANED_JD) AS CLEANED_JD_LENGTH,
        REGEXP_COUNT(CLEANED_JD, '[^\x00-\x7F]') AS NON_ASCII_CHAR_COUNT,
        ROUND(
            REGEXP_COUNT(CLEANED_JD, '[^\x00-\x7F]') * 100.0
            / NULLIF(LENGTH(CLEANED_JD), 0),
            2
        ) AS NON_ASCII_CHAR_PCT
    FROM IDENTIFIER($CANONICAL_TABLE)
    WHERE CLEANED_JD IS NOT NULL
      AND TRIM(CLEANED_JD) <> ''
),
classified AS (
    SELECT
        *,
        IFF(
            CLEANED_JD_LENGTH >= $TRANSLATION_MIN_CLEANED_JD_LENGTH
            AND (
                   NON_ASCII_CHAR_COUNT >= $TRANSLATION_MIN_NON_ASCII_COUNT
                OR NON_ASCII_CHAR_PCT >= $TRANSLATION_MIN_NON_ASCII_PCT
            ),
            TRUE,
            FALSE
        ) AS LIKELY_NON_ENGLISH_TRANSLATION_CANDIDATE
    FROM scored
)

SELECT
    COUNT(*) AS FINAL_CANONICAL_ROWS,
    COUNT_IF(LIKELY_NON_ENGLISH_TRANSLATION_CANDIDATE) AS LIKELY_NON_ENGLISH_TRANSLATION_CANDIDATES,
    COUNT_IF(NON_ASCII_CHAR_COUNT > 0) AS ROWS_WITH_ANY_NON_ASCII,
    COUNT_IF(CLEANED_JD_LENGTH < $TRANSLATION_MIN_CLEANED_JD_LENGTH) AS LOW_CONTENT_OR_SHORT_JD_ROWS,
    ROUND(AVG(CLEANED_JD_LENGTH), 2) AS AVG_CLEANED_JD_LENGTH,
    ROUND(AVG(NON_ASCII_CHAR_PCT), 2) AS AVG_NON_ASCII_CHAR_PCT
FROM classified;


In [ ]:
%%sql -r dataframe_15
-- SQL Cell 9D: Optional post-canonical translation enrichment.
-- This translates only final canonical rows that are likely non-English.
-- By default ENABLE_CANONICAL_TRANSLATION = FALSE to prevent accidental Cortex credit usage.
-- To run translation, set ENABLE_CANONICAL_TRANSLATION = TRUE in the parameter cell and rerun this cell.

CREATE OR REPLACE TABLE IDENTIFIER($CANONICAL_TRANSLATION_TABLE) AS
WITH scored AS (
    SELECT
        CANONICAL_JOB_ID,
        DESCRIPTION_SOURCE_JOB_ID,
        TITLE_RAW,
        TITLE_TRANSLATED,
        COMPANY,
        COUNTRY,
        STATE,
        CANONICAL_POST_DATE,
        CANONICALIZATION_ACTION,
        DEDUPE_CRITERIA,
        SOURCE_RECORD_COUNT,
        CLEANED_JD,
        LENGTH(CLEANED_JD) AS CLEANED_JD_LENGTH,
        REGEXP_COUNT(CLEANED_JD, '[^\x00-\x7F]') AS NON_ASCII_CHAR_COUNT,
        ROUND(
            REGEXP_COUNT(CLEANED_JD, '[^\x00-\x7F]') * 100.0
            / NULLIF(LENGTH(CLEANED_JD), 0),
            2
        ) AS NON_ASCII_CHAR_PCT
    FROM IDENTIFIER($CANONICAL_TABLE)
    WHERE CLEANED_JD IS NOT NULL
      AND TRIM(CLEANED_JD) <> ''
),
translation_candidates AS (
    SELECT
        *,
        CASE
            WHEN CLEANED_JD_LENGTH < $TRANSLATION_MIN_CLEANED_JD_LENGTH
                THEN 'SHORT_OR_LOW_CONTENT_JD_NOT_TRANSLATED'
            WHEN NON_ASCII_CHAR_COUNT >= $TRANSLATION_MIN_NON_ASCII_COUNT
              OR NON_ASCII_CHAR_PCT >= $TRANSLATION_MIN_NON_ASCII_PCT
                THEN 'LIKELY_NON_ENGLISH_BY_NON_ASCII_SIGNAL'
            ELSE 'NOT_SELECTED_FOR_TRANSLATION'
        END AS TRANSLATION_CANDIDATE_REASON
    FROM scored
    WHERE CLEANED_JD_LENGTH >= $TRANSLATION_MIN_CLEANED_JD_LENGTH
      AND (
             NON_ASCII_CHAR_COUNT >= $TRANSLATION_MIN_NON_ASCII_COUNT
          OR NON_ASCII_CHAR_PCT >= $TRANSLATION_MIN_NON_ASCII_PCT
      )
      AND $ENABLE_CANONICAL_TRANSLATION = TRUE
)

SELECT
    CANONICAL_JOB_ID,
    DESCRIPTION_SOURCE_JOB_ID,
    TITLE_RAW,
    TITLE_TRANSLATED,
    COMPANY,
    COUNTRY,
    STATE,
    CANONICAL_POST_DATE,
    CANONICALIZATION_ACTION,
    DEDUPE_CRITERIA,
    SOURCE_RECORD_COUNT,
    CLEANED_JD_LENGTH,
    NON_ASCII_CHAR_COUNT,
    NON_ASCII_CHAR_PCT,
    TRANSLATION_CANDIDATE_REASON,
    IFF(CLEANED_JD_LENGTH > $TRANSLATION_TEXT_CHAR_LIMIT, TRUE, FALSE) AS TRANSLATION_TEXT_TRUNCATED_FLAG,
    LEFT(CLEANED_JD, $TRANSLATION_TEXT_CHAR_LIMIT) AS CLEANED_JD_TRANSLATION_INPUT,
    SNOWFLAKE.CORTEX.TRANSLATE(
        LEFT(CLEANED_JD, $TRANSLATION_TEXT_CHAR_LIMIT),
        '',
        'en'
    ) AS CLEANED_JD_TRANSLATED_TO_ENGLISH
FROM translation_candidates;

SELECT
    COUNT(*) AS TRANSLATED_CANONICAL_ROWS,
    COUNT_IF(TRANSLATION_TEXT_TRUNCATED_FLAG) AS TRANSLATION_TEXT_TRUNCATED_ROWS
FROM IDENTIFIER($CANONICAL_TRANSLATION_TABLE);


In [ ]:
%%sql -r dataframe_23
select * from PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_CANONICAL_TRANSLATION_UPRCID_TITLETR_10D limit 500;

In [ ]:
%%sql -r dataframe_16
-- SQL Cell 11A: Create handoff spot-check samples from each decision bucket.

CREATE OR REPLACE TRANSIENT TABLE IDENTIFIER($BUCKET_SPOTCHECK_TABLE) AS
WITH candidate_samples AS (
    SELECT
        '01_MERGE_EXACT_JD' AS SAMPLE_BUCKET,
        c.*
    FROM IDENTIFIER($CANONICAL_TABLE) c
    WHERE CANONICALIZATION_ACTION = 'MERGE_EXACT_JD'

    UNION ALL

    SELECT
        '02_MERGE_TOKEN_CONTAINMENT' AS SAMPLE_BUCKET,
        c.*
    FROM IDENTIFIER($CANONICAL_TABLE) c
    WHERE CANONICALIZATION_ACTION = 'MERGE_TOKEN_CONTAINMENT_USE_BEST_JD'

    UNION ALL

    SELECT
        '03_REVIEW_BUCKET' AS SAMPLE_BUCKET,
        c.*
    FROM IDENTIFIER($CANONICAL_TABLE) c
    WHERE CANONICALIZATION_ACTION = 'REVIEW_BUCKET'

    UNION ALL

    SELECT
        '04_KEEP_SEPARATE' AS SAMPLE_BUCKET,
        c.*
    FROM IDENTIFIER($CANONICAL_TABLE) c
    WHERE CANONICALIZATION_ACTION = 'KEEP_SEPARATE_MATERIAL_JD_CHANGE'

    UNION ALL

    SELECT
        '05_MULTI_REGION' AS SAMPLE_BUCKET,
        c.*
    FROM IDENTIFIER($CANONICAL_TABLE) c
    WHERE IS_MULTIPLE_COUNTRY_POSTING = TRUE
       OR IS_MULTIPLE_STATE_POSTING = TRUE

    UNION ALL

    SELECT
        '06_PAIR_DIAGNOSTIC_SKIPPED_REVIEW' AS SAMPLE_BUCKET,
        c.*
    FROM IDENTIFIER($CANONICAL_TABLE) c
    WHERE COALESCE(PAIR_DIAGNOSTIC_SKIPPED_FLAG, FALSE) = TRUE
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY SAMPLE_BUCKET
            ORDER BY
                SOURCE_RECORD_COUNT DESC,
                COALESCE(MAX_RELATED_TOKEN_CONTAINMENT_SCORE, 0) DESC,
                CANONICAL_POST_DATE,
                CANONICAL_JOB_ID
        ) AS SAMPLE_RANK
    FROM candidate_samples
)
SELECT *
FROM ranked
WHERE SAMPLE_RANK <= 5;

SELECT
    SAMPLE_BUCKET,
    SAMPLE_RANK,
    CANONICAL_JOB_ID,
    CANONICAL_POST_DATE,
    COMPANY,
    TITLE_RAW,
    COUNTRY,
    STATE,
    CANONICALIZATION_ACTION,
    DEDUPE_CRITERIA,
    SOURCE_RECORD_COUNT,
    JOB_IDS_IN_CLUSTER,
    ALL_RELEVANT_JOB_IDS,
    CANDIDATE_RELEVANT_JOB_IDS,
    IS_MULTIPLE_COUNTRY_POSTING,
    IS_MULTIPLE_STATE_POSTING,
    PAIR_DIAGNOSTIC_SKIPPED_FLAG,
    PAIR_DIAGNOSTIC_SKIP_REASON,
    MAX_RELATED_TOKEN_CONTAINMENT_SCORE,
    LEFT(CLEANED_JD, 750) AS CLEANED_JD_SAMPLE
FROM IDENTIFIER($BUCKET_SPOTCHECK_TABLE)
ORDER BY SAMPLE_BUCKET, SAMPLE_RANK;


In [ ]:
%%sql -r dataframe_17
-- SQL Cell 10: Create temporary lineage and review tables for inspection.

CREATE OR REPLACE TRANSIENT TABLE IDENTIFIER($LINEAGE_TABLE) AS
SELECT
    c.CANONICAL_JOB_ID,
    c.CLUSTER_ID,
    c.MERGE_GROUP_ID,
    c.CANONICALIZATION_ACTION,
    r.JOB_ID AS ORIGINAL_JOB_ID,
    r.POST_DAY AS ORIGINAL_POST_DATE,
    r.SOURCE_SIGNATURE AS ORIGINAL_SOURCE_SIGNATURE,
    r.JOIN_ACTION,
    r.JOIN_REASON
FROM IDENTIFIER($CANONICAL_TABLE) c
JOIN IDENTIFIER($MERGE_GROUP_TABLE) r
  ON c.MERGE_GROUP_ID = r.MERGE_GROUP_ID;

CREATE OR REPLACE TRANSIENT TABLE IDENTIFIER($REVIEW_TABLE) AS
SELECT *
FROM IDENTIFIER($CANONICAL_TABLE)
WHERE CANONICALIZATION_ACTION = 'REVIEW_BUCKET';

SELECT
    CANONICALIZATION_ACTION,
    COUNT(*) AS LINEAGE_ROWS,
    COUNT(DISTINCT CANONICAL_JOB_ID) AS OUTPUT_ROWS,
    COUNT(DISTINCT ORIGINAL_JOB_ID) AS ORIGINAL_JOB_IDS
FROM IDENTIFIER($LINEAGE_TABLE)
GROUP BY CANONICALIZATION_ACTION
ORDER BY LINEAGE_ROWS DESC;


In [ ]:
%%sql -r dataframe_18
-- SQL Cell 11: Preview final canonical output.

SELECT
    CANONICAL_JOB_ID,
    CANONICAL_POST_DATE,
    COMPANY,
    TITLE_RAW,
    COUNTRY,
    STATE,
    CANONICALIZATION_ACTION,
    DEDUPE_CRITERIA,
    SOURCE_RECORD_COUNT,
    JOB_IDS_IN_CLUSTER,
    CANDIDATE_CLUSTER_JOB_IDS,
    ALL_RELEVANT_JOB_IDS,
    CANDIDATE_RELEVANT_JOB_IDS,
    ALL_COUNTRY_POSTED,
    ALL_STATE_POSTED,
    IS_MULTIPLE_COUNTRY_POSTING,
    IS_MULTIPLE_STATE_POSTING,
    SOURCE_SIGNATURE_LIST,
    SOURCE_LINKEDIN,
    SOURCE_INDEED,
    SOURCE_COMPANY_SITES,
    SOURCE_OTHER_AGGREGATORS,
    MAX_RELATED_TOKEN_CONTAINMENT_SCORE,
    LEFT(CLEANED_JD, 1000) AS CLEANED_JD_SAMPLE,
    LEFT(SELECTED_DESCRIPTION, 1000) AS SELECTED_DESCRIPTION_SAMPLE
FROM IDENTIFIER($CANONICAL_TABLE)
ORDER BY COMPANY, TITLE_RAW, CANONICAL_POST_DATE, CANONICAL_JOB_ID
LIMIT 1000;


In [ ]:
%%sql -r dataframe_19
SELECT
    CANONICALIZATION_ACTION,
    DEDUPE_CRITERIA,
    COUNT(*) AS CANONICAL_ROWS,
    SUM(
        CASE
            WHEN ALL_RELEVANT_JOB_IDS IS NULL OR TRIM(ALL_RELEVANT_JOB_IDS) = '' THEN 0
            ELSE ARRAY_SIZE(SPLIT(ALL_RELEVANT_JOB_IDS, ','))
        END
    ) AS SOURCE_JOB_IDS_REPRESENTED
FROM PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_CANONICAL_OUTPUT_UPRCID_TITLETR_TC_GROUP_REP_10D
GROUP BY
    CANONICALIZATION_ACTION,
    DEDUPE_CRITERIA
ORDER BY
    CANONICAL_ROWS DESC;

In [ ]:
%%sql -r dataframe_20
WITH canonical_enriched AS (
    SELECT
        c.*,

        CASE
            WHEN ALL_RELEVANT_JOB_IDS IS NULL OR TRIM(ALL_RELEVANT_JOB_IDS) = '' THEN 0
            ELSE ARRAY_SIZE(SPLIT(ALL_RELEVANT_JOB_IDS, ','))
        END AS RELEVANT_JOB_ID_COUNT,

        LENGTH(CLEANED_JD) AS CLEANED_JD_LENGTH,

        LEFT(CLEANED_JD, 1000) AS CLEANED_JD_HEAD_1000,

        SUBSTR(
            CLEANED_JD,
            GREATEST(1, FLOOR(LENGTH(CLEANED_JD) / 2) - 500),
            1000
        ) AS CLEANED_JD_MIDDLE_1000,

        RIGHT(CLEANED_JD, 1000) AS CLEANED_JD_TAIL_1000

    FROM PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_CANONICAL_OUTPUT_UPRCID_TITLETR_TC_GROUP_REP_10D c
),

bucketed AS (

    SELECT
        '01_MERGE_EXACT_JD' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'MERGE_EXACT_JD'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY HASH(CANONICAL_JOB_ID)
    ) <= 2000

    UNION ALL

    SELECT
        '02_MERGE_TOKEN_CONTAINMENT' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'MERGE_TOKEN_CONTAINMENT_USE_BEST_JD'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY HASH(CANONICAL_JOB_ID)
    ) <= 2000

    UNION ALL

    SELECT
        '03_REVIEW_BUCKET' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'REVIEW_BUCKET'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY HASH(CANONICAL_JOB_ID)
    ) <= 2000

    UNION ALL

    SELECT
        '04_KEEP_SEPARATE' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'KEEP_SEPARATE'
       OR DEDUPE_CRITERIA ILIKE '%KEEP_SEPARATE%'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY HASH(CANONICAL_JOB_ID)
    ) <= 2000

    UNION ALL

    SELECT
        '05_NOTHING_NO_DUPLICATE' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'NOTHING'
       OR DEDUPE_CRITERIA = 'NOTHING'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY HASH(CANONICAL_JOB_ID)
    ) <= 2000

    UNION ALL

    SELECT
        '06_MULTI_REGION' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE COALESCE(IS_MULTIPLE_COUNTRY_POSTING, FALSE) = TRUE
       OR COALESCE(IS_MULTIPLE_STATE_POSTING, FALSE) = TRUE
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)
    ) <= 2000

    UNION ALL

    SELECT
        '07_LARGEST_MERGE_GROUPS' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE RELEVANT_JOB_ID_COUNT > 1
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)
    ) <= 2000

    UNION ALL

    SELECT
        '08_PAIR_DIAGNOSTIC_SKIPPED_REVIEW' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE COALESCE(PAIR_DIAGNOSTIC_SKIPPED_FLAG, FALSE) = TRUE
      AND CANONICALIZATION_ACTION = 'REVIEW_BUCKET'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)
    ) <= 2000
)

SELECT
    SAMPLE_BUCKET,

    CANONICAL_JOB_ID,
    CLUSTER_ID,
    MERGE_GROUP_ID,

    TITLE_RAW,
    COMPANY,
    POST_DATE,
    COUNTRY,
    STATE,

    CANONICALIZATION_ACTION,
    DEDUPE_CRITERIA,
    CANONICALIZATION_REASON,

    RELEVANT_JOB_ID_COUNT,
    ALL_RELEVANT_JOB_IDS,
    CANDIDATE_RELEVANT_JOB_IDS,

    ALL_COUNTRY_POSTED,
    ALL_STATE_POSTED,
    IS_MULTIPLE_COUNTRY_POSTING,
    IS_MULTIPLE_STATE_POSTING,

    PAIR_DIAGNOSTIC_SKIPPED_FLAG,
    PAIR_DIAGNOSTIC_SKIP_REASON,
    PAIR_DIAGNOSTIC_SKIPPED_CLUSTER_ROWS,
    PAIR_DIAGNOSTIC_SKIPPED_THEORETICAL_PAIRS,

    SOURCE_LINKEDIN,
    SOURCE_INDEED,
    SOURCE_COMPANY_SITES,
    SOURCE_STAFFINGFIRMS,
    SOURCE_REGIONAL_AGGREGATORS,
    SOURCE_OTHER_AGGREGATORS,

    CLEANED_JD_LENGTH,
    CLEANED_JD_HEAD_1000,
    CLEANED_JD_MIDDLE_1000,
    CLEANED_JD_TAIL_1000

FROM bucketed
ORDER BY
    SAMPLE_BUCKET,
    TITLE_RAW,
    COMPANY,
    POST_DATE;

In [ ]:
%%sql -r dataframe_21
WITH canonical_enriched AS (
    SELECT
        c.*,
        CASE
            WHEN ALL_RELEVANT_JOB_IDS IS NULL OR TRIM(ALL_RELEVANT_JOB_IDS) = '' THEN 0
            ELSE ARRAY_SIZE(SPLIT(ALL_RELEVANT_JOB_IDS, ','))
        END AS RELEVANT_JOB_ID_COUNT
    FROM PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_CANONICAL_OUTPUT_UPRCID_TITLETR_TC_GROUP_REP_10D c
),

bucketed AS (

    SELECT '01_MERGE_EXACT_JD' AS SAMPLE_BUCKET, CANONICAL_JOB_ID, RELEVANT_JOB_ID_COUNT
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'MERGE_EXACT_JD'
    QUALIFY ROW_NUMBER() OVER (ORDER BY HASH(CANONICAL_JOB_ID)) <= 2000

    UNION ALL

    SELECT '02_MERGE_TOKEN_CONTAINMENT' AS SAMPLE_BUCKET, CANONICAL_JOB_ID, RELEVANT_JOB_ID_COUNT
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'MERGE_TOKEN_CONTAINMENT_USE_BEST_JD'
    QUALIFY ROW_NUMBER() OVER (ORDER BY HASH(CANONICAL_JOB_ID)) <= 2000

    UNION ALL

    SELECT '03_REVIEW_BUCKET' AS SAMPLE_BUCKET, CANONICAL_JOB_ID, RELEVANT_JOB_ID_COUNT
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'REVIEW_BUCKET'
    QUALIFY ROW_NUMBER() OVER (ORDER BY HASH(CANONICAL_JOB_ID)) <= 2000

    UNION ALL

    SELECT '04_KEEP_SEPARATE' AS SAMPLE_BUCKET, CANONICAL_JOB_ID, RELEVANT_JOB_ID_COUNT
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'KEEP_SEPARATE'
       OR DEDUPE_CRITERIA ILIKE '%KEEP_SEPARATE%'
    QUALIFY ROW_NUMBER() OVER (ORDER BY HASH(CANONICAL_JOB_ID)) <= 2000

    UNION ALL

    SELECT '05_NOTHING_NO_DUPLICATE' AS SAMPLE_BUCKET, CANONICAL_JOB_ID, RELEVANT_JOB_ID_COUNT
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'NOTHING'
       OR DEDUPE_CRITERIA = 'NOTHING'
    QUALIFY ROW_NUMBER() OVER (ORDER BY HASH(CANONICAL_JOB_ID)) <= 2000

    UNION ALL

    SELECT '06_MULTI_REGION' AS SAMPLE_BUCKET, CANONICAL_JOB_ID, RELEVANT_JOB_ID_COUNT
    FROM canonical_enriched
    WHERE COALESCE(IS_MULTIPLE_COUNTRY_POSTING, FALSE) = TRUE
       OR COALESCE(IS_MULTIPLE_STATE_POSTING, FALSE) = TRUE
    QUALIFY ROW_NUMBER() OVER (ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)) <= 2000

    UNION ALL

    SELECT '07_LARGEST_MERGE_GROUPS' AS SAMPLE_BUCKET, CANONICAL_JOB_ID, RELEVANT_JOB_ID_COUNT
    FROM canonical_enriched
    WHERE RELEVANT_JOB_ID_COUNT > 1
    QUALIFY ROW_NUMBER() OVER (ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)) <= 2000

    UNION ALL

    SELECT '08_PAIR_DIAGNOSTIC_SKIPPED_REVIEW' AS SAMPLE_BUCKET, CANONICAL_JOB_ID, RELEVANT_JOB_ID_COUNT
    FROM canonical_enriched
    WHERE COALESCE(PAIR_DIAGNOSTIC_SKIPPED_FLAG, FALSE) = TRUE
      AND CANONICALIZATION_ACTION = 'REVIEW_BUCKET'
    QUALIFY ROW_NUMBER() OVER (ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)) <= 2000
)

SELECT
    SAMPLE_BUCKET,
    COUNT(*) AS SAMPLE_ROWS,
    COUNT(DISTINCT CANONICAL_JOB_ID) AS DISTINCT_CANONICAL_JOB_IDS,
    SUM(RELEVANT_JOB_ID_COUNT) AS SOURCE_JOB_IDS_REPRESENTED,
    MIN(RELEVANT_JOB_ID_COUNT) AS MIN_RELEVANT_JOB_ID_COUNT,
    MAX(RELEVANT_JOB_ID_COUNT) AS MAX_RELEVANT_JOB_ID_COUNT,
    ROUND(AVG(RELEVANT_JOB_ID_COUNT), 2) AS AVG_RELEVANT_JOB_ID_COUNT
FROM bucketed
GROUP BY SAMPLE_BUCKET
ORDER BY SAMPLE_BUCKET;

In [ ]:
%%sql -r dataframe_22
WITH canonical_enriched AS (
    SELECT
        c.*,

        CASE
            WHEN ALL_RELEVANT_JOB_IDS IS NULL OR TRIM(ALL_RELEVANT_JOB_IDS) = '' THEN 0
            ELSE ARRAY_SIZE(SPLIT(ALL_RELEVANT_JOB_IDS, ','))
        END AS RELEVANT_JOB_ID_COUNT,

        LENGTH(CLEANED_JD) AS CANONICAL_CLEANED_JD_LENGTH,

        LEFT(CLEANED_JD, 1000) AS CANONICAL_CLEANED_JD_HEAD_1000,

        SUBSTR(
            CLEANED_JD,
            GREATEST(1, FLOOR(LENGTH(CLEANED_JD) / 2) - 500),
            1000
        ) AS CANONICAL_CLEANED_JD_MIDDLE_1000,

        RIGHT(CLEANED_JD, 1000) AS CANONICAL_CLEANED_JD_TAIL_1000

    FROM PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_CANONICAL_OUTPUT_UPRCID_TITLETR_TC_GROUP_REP_10D c
),

sampled_canonical AS (

    SELECT
        '01_MERGE_EXACT_JD' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'MERGE_EXACT_JD'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)
    ) <= 300

    UNION ALL

    SELECT
        '02_MERGE_TOKEN_CONTAINMENT' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'MERGE_TOKEN_CONTAINMENT_USE_BEST_JD'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)
    ) <= 300

    UNION ALL

    SELECT
        '03_REVIEW_BUCKET' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'REVIEW_BUCKET'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)
    ) <= 300

    UNION ALL

    SELECT
        '04_KEEP_SEPARATE' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE CANONICALIZATION_ACTION = 'KEEP_SEPARATE'
       OR DEDUPE_CRITERIA ILIKE '%KEEP_SEPARATE%'
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)
    ) <= 300

    UNION ALL

    SELECT
        '05_MULTI_REGION' AS SAMPLE_BUCKET,
        *
    FROM canonical_enriched
    WHERE COALESCE(IS_MULTIPLE_COUNTRY_POSTING, FALSE) = TRUE
       OR COALESCE(IS_MULTIPLE_STATE_POSTING, FALSE) = TRUE
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY RELEVANT_JOB_ID_COUNT DESC, HASH(CANONICAL_JOB_ID)
    ) <= 300
),

exploded AS (
    SELECT
        s.*,
        TRIM(f.value::STRING) AS ORIGINAL_JOB_ID,
        ROW_NUMBER() OVER (
            PARTITION BY s.SAMPLE_BUCKET, s.CANONICAL_JOB_ID
            ORDER BY TRIM(f.value::STRING)
        ) AS ORIGINAL_ROW_SAMPLE_RN
    FROM sampled_canonical s,
         LATERAL FLATTEN(INPUT => SPLIT(s.ALL_RELEVANT_JOB_IDS, ',')) f
),

limited_exploded AS (
    SELECT *
    FROM exploded
    WHERE ORIGINAL_ROW_SAMPLE_RN <= 10
),

source_clean AS (
    SELECT
        TO_VARCHAR(JOB_ID) AS ORIGINAL_JOB_ID,
        POST_DATE AS ORIGINAL_POST_DATE,
        TITLE_RAW AS ORIGINAL_TITLE_RAW,
        COMPANY AS ORIGINAL_COMPANY,
        COUNTRY AS ORIGINAL_COUNTRY,
        STATE AS ORIGINAL_STATE,

        SOURCE_LINKEDIN AS ORIGINAL_SOURCE_LINKEDIN,
        SOURCE_INDEED AS ORIGINAL_SOURCE_INDEED,
        SOURCE_COMPANY_SITES AS ORIGINAL_SOURCE_COMPANY_SITES,
        SOURCE_STAFFINGFIRMS AS ORIGINAL_SOURCE_STAFFINGFIRMS,
        SOURCE_REGIONAL_AGGREGATORS AS ORIGINAL_SOURCE_REGIONAL_AGGREGATORS,
        SOURCE_OTHER_AGGREGATORS AS ORIGINAL_SOURCE_OTHER_AGGREGATORS,

        LENGTH(
            LOWER(
                TRIM(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            COALESCE(DESCRIPTION, ''),
                            '<[^>]+>',
                            ' '
                        ),
                        '[[:space:]]+',
                        ' '
                    )
                )
            )
        ) AS ORIGINAL_CLEANED_JD_LENGTH,

        LEFT(
            LOWER(
                TRIM(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            COALESCE(DESCRIPTION, ''),
                            '<[^>]+>',
                            ' '
                        ),
                        '[[:space:]]+',
                        ' '
                    )
                )
            ),
            1000
        ) AS ORIGINAL_CLEANED_JD_HEAD_1000,

        SUBSTR(
            LOWER(
                TRIM(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            COALESCE(DESCRIPTION, ''),
                            '<[^>]+>',
                            ' '
                        ),
                        '[[:space:]]+',
                        ' '
                    )
                )
            ),
            GREATEST(
                1,
                FLOOR(
                    LENGTH(
                        LOWER(
                            TRIM(
                                REGEXP_REPLACE(
                                    REGEXP_REPLACE(
                                        COALESCE(DESCRIPTION, ''),
                                        '<[^>]+>',
                                        ' '
                                    ),
                                    '[[:space:]]+',
                                    ' '
                                )
                            )
                        )
                    ) / 2
                ) - 500
            ),
            1000
        ) AS ORIGINAL_CLEANED_JD_MIDDLE_1000,

        RIGHT(
            LOWER(
                TRIM(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            COALESCE(DESCRIPTION, ''),
                            '<[^>]+>',
                            ' '
                        ),
                        '[[:space:]]+',
                        ' '
                    )
                )
            ),
            1000
        ) AS ORIGINAL_CLEANED_JD_TAIL_1000

    FROM H_EDW.HS_REVELIO_EDW_V.V_POSTINGS_UNIFIED_INDIVIDUAL
    WHERE POST_DATE >= TO_DATE('2025-01-01')
      AND POST_DATE <  TO_DATE('2025-04-01')
)

SELECT
    e.SAMPLE_BUCKET,

    e.CANONICAL_JOB_ID,
    e.MERGE_GROUP_ID,
    e.CLUSTER_ID,

    e.CANONICALIZATION_ACTION,
    e.DEDUPE_CRITERIA,
    e.CANONICALIZATION_REASON,

    e.RELEVANT_JOB_ID_COUNT,
    e.ALL_RELEVANT_JOB_IDS,
    e.CANDIDATE_RELEVANT_JOB_IDS,

    e.TITLE_RAW AS CANONICAL_TITLE_RAW,
    e.COMPANY AS CANONICAL_COMPANY,
    e.POST_DATE AS CANONICAL_POST_DATE,
    e.COUNTRY AS CANONICAL_COUNTRY,
    e.STATE AS CANONICAL_STATE,

    e.ALL_COUNTRY_POSTED,
    e.ALL_STATE_POSTED,
    e.IS_MULTIPLE_COUNTRY_POSTING,
    e.IS_MULTIPLE_STATE_POSTING,

    e.CANONICAL_CLEANED_JD_LENGTH,
    e.CANONICAL_CLEANED_JD_HEAD_1000,
    e.CANONICAL_CLEANED_JD_MIDDLE_1000,
    e.CANONICAL_CLEANED_JD_TAIL_1000,

    e.ORIGINAL_ROW_SAMPLE_RN,
    e.ORIGINAL_JOB_ID,

    sc.ORIGINAL_POST_DATE,
    sc.ORIGINAL_TITLE_RAW,
    sc.ORIGINAL_COMPANY,
    sc.ORIGINAL_COUNTRY,
    sc.ORIGINAL_STATE,

    sc.ORIGINAL_SOURCE_LINKEDIN,
    sc.ORIGINAL_SOURCE_INDEED,
    sc.ORIGINAL_SOURCE_COMPANY_SITES,
    sc.ORIGINAL_SOURCE_STAFFINGFIRMS,
    sc.ORIGINAL_SOURCE_REGIONAL_AGGREGATORS,
    sc.ORIGINAL_SOURCE_OTHER_AGGREGATORS,

    sc.ORIGINAL_CLEANED_JD_LENGTH,
    sc.ORIGINAL_CLEANED_JD_HEAD_1000,
    sc.ORIGINAL_CLEANED_JD_MIDDLE_1000,
    sc.ORIGINAL_CLEANED_JD_TAIL_1000

FROM limited_exploded e
LEFT JOIN source_clean sc
  ON e.ORIGINAL_JOB_ID = sc.ORIGINAL_JOB_ID
ORDER BY
    e.SAMPLE_BUCKET,
    e.CANONICAL_JOB_ID,
    e.ORIGINAL_ROW_SAMPLE_RN;

In [ ]:
%%sql -r dataframe_24
-- ============================================================
-- REFRESH-DAY WINDOW ANALYSIS
-- Goal: find how many days you must hold a posting before
--       finalizing it, so that late-arriving duplicates/near-dups
--       have (almost) all landed and been merged.
--
-- Run against the persistent canonical output produced by the pipeline.
-- No pipeline changes required.
-- ============================================================

SET CANON = 'PC_FIVETRAN_DB.DA_WORKSPACE.COSMOS_DEDUP_Q1_2025_CANONICAL_OUTPUT_UPRCID_TITLETR_TC_GROUP_REP_10D';

-- ------------------------------------------------------------
-- 1) SPAN DISTRIBUTION OF *MERGED* GROUPS ONLY
--    (groups that actually absorbed >1 source record).
--    MERGE_GROUP_SPAN_DAYS = days between first and last posting
--    that ended up in the same canonical record.
--    This IS the "how long duplicates keep arriving" curve.
-- ------------------------------------------------------------
WITH merged AS (
    SELECT
        MERGE_GROUP_SPAN_DAYS,
        SOURCE_RECORD_COUNT
    FROM IDENTIFIER($CANON)
    WHERE SOURCE_RECORD_COUNT > 1          -- only groups that merged something
      AND CANONICALIZATION_ACTION LIKE 'MERGE%'
)
SELECT
    'MERGED_GROUP_SPAN_PERCENTILES' AS METRIC,
    COUNT(*)                                              AS MERGED_GROUPS,
    ROUND(AVG(MERGE_GROUP_SPAN_DAYS), 2)                  AS AVG_SPAN_DAYS,
    MEDIAN(MERGE_GROUP_SPAN_DAYS)                         AS P50_SPAN_DAYS,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY MERGE_GROUP_SPAN_DAYS) AS P75_SPAN_DAYS,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY MERGE_GROUP_SPAN_DAYS) AS P90_SPAN_DAYS,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY MERGE_GROUP_SPAN_DAYS) AS P95_SPAN_DAYS,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY MERGE_GROUP_SPAN_DAYS) AS P99_SPAN_DAYS,
    MAX(MERGE_GROUP_SPAN_DAYS)                            AS MAX_SPAN_DAYS
FROM merged;

-- ------------------------------------------------------------
-- 2) CUMULATIVE "CAPTURE CURVE"
--    For each candidate hold window W (0..35 days), what % of all
--    merged groups would have been FULLY captured if we finalized
--    W days after the group's FIRST posting day?
--    Read this as: "hold W days -> capture X% of merges."
-- ------------------------------------------------------------
WITH merged AS (
    SELECT MERGE_GROUP_SPAN_DAYS
    FROM IDENTIFIER($CANON)
    WHERE SOURCE_RECORD_COUNT > 1
      AND CANONICALIZATION_ACTION LIKE 'MERGE%'
),
windows AS (
    SELECT SEQ4() AS W
    FROM TABLE(GENERATOR(ROWCOUNT => 36))   -- windows 0..35 days
),
total AS (SELECT COUNT(*) AS N FROM merged)
SELECT
    w.W                                          AS HOLD_WINDOW_DAYS,
    COUNT_IF(m.MERGE_GROUP_SPAN_DAYS <= w.W)     AS GROUPS_FULLY_CAPTURED,
    (SELECT N FROM total)                        AS TOTAL_MERGED_GROUPS,
    ROUND(100.0 * COUNT_IF(m.MERGE_GROUP_SPAN_DAYS <= w.W)
          / NULLIF((SELECT N FROM total), 0), 3) AS PCT_MERGES_CAPTURED
FROM windows w
CROSS JOIN merged m
GROUP BY w.W
ORDER BY w.W;

-- ------------------------------------------------------------
-- 3) MARGINAL VALUE OF EACH EXTRA DAY
--    Where does the capture curve flatten? The "best" window is
--    the knee: the smallest W where extra days add < ~0.1-0.5%.
-- ------------------------------------------------------------
WITH merged AS (
    SELECT MERGE_GROUP_SPAN_DAYS
    FROM IDENTIFIER($CANON)
    WHERE SOURCE_RECORD_COUNT > 1
      AND CANONICALIZATION_ACTION LIKE 'MERGE%'
),
per_span AS (
    SELECT MERGE_GROUP_SPAN_DAYS AS SPAN, COUNT(*) AS GROUPS_CLOSING_ON_DAY
    FROM merged
    GROUP BY MERGE_GROUP_SPAN_DAYS
),
total AS (SELECT COUNT(*) AS N FROM merged)
SELECT
    SPAN                                              AS GROUP_SPAN_DAYS,
    GROUPS_CLOSING_ON_DAY,
    ROUND(100.0 * GROUPS_CLOSING_ON_DAY
          / NULLIF((SELECT N FROM total),0), 3)       AS PCT_OF_MERGES_THAT_NEED_THIS_MANY_DAYS
FROM per_span
ORDER BY SPAN;